## 드라이브 연결 및 설정

In [1]:
!pip install kiwipiepy
!pip install transformers torch pandas

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## corpus

In [2]:
from pathlib import Path
import re
import html
import unicodedata
import pandas as pd


# RAW_XML_DIR = Path("data/dart/raw_xml")
# COMPANY_MASTER_PATH = Path("data/company_master.csv")

RAW_XML_DIR = Path("/content/drive/MyDrive/raw_xml")
COMPANY_MASTER_PATH = Path("/content/drive/MyDrive/company_master.csv")

TARGET_TITLE_REGEX = {
    "II. 사업의 내용": r"^(II|Ⅱ)\.\s*사업의\s*내용",
    "IV. 이사의 경영진단 및 분석의견": r"^(IV|Ⅳ)\.\s*이사의\s*경영진단\s*및\s*분석의견",
    "VI. 이사회 등 회사의 기관에 관한 사항": r"^(VI|Ⅵ)\.\s*이사회\s*등\s*회사의\s*기관에\s*관한\s*사항",
}

TITLE_RE = re.compile(r"<TITLE\b[^>]*>(.*?)</TITLE>", flags=re.I | re.S)
MAIN_TITLE_RE = re.compile(
    r"^\s*(I|II|III|IV|V|VI|VII|VIII|IX|X|Ⅰ|Ⅱ|Ⅲ|Ⅳ|Ⅴ|Ⅵ|Ⅶ|Ⅷ|Ⅸ|Ⅹ)\."
)


def clean_xml_text(text: str) -> str:
    text = "" if text is None else str(text)
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_report_text(text: str) -> str:
    text = clean_xml_text(text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_xml_filename(path: Path) -> tuple[str, int, str]:
    match = re.match(r"(\d{6})_(\d{4})_(\d+)\.xml$", path.name)
    if not match:
        raise ValueError(f"Unexpected XML filename: {path.name}")

    stock_code, fiscal_year, rcept_no = match.groups()
    return stock_code, int(fiscal_year), rcept_no


def extract_target_sections(xml_text: str) -> list[dict]:
    titles = []

    for match in TITLE_RE.finditer(xml_text):
        title = clean_xml_text(match.group(1))
        if MAIN_TITLE_RE.match(title):
            titles.append((title, match.start()))

    sections = []

    for i, (title, start) in enumerate(titles):
        section_name = None

        for name, pattern in TARGET_TITLE_REGEX.items():
            if re.search(pattern, title):
                section_name = name
                break

        if section_name is None:
            continue

        end = titles[i + 1][1] if i + 1 < len(titles) else len(xml_text)
        section_raw = xml_text[start:end]

        sections.append({
            "section": section_name,
            "text": clean_xml_text(section_raw),
        })

    return sections


def load_company_names(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=["stock_code", "company_name"])

    company_df = pd.read_csv(path, dtype={"stock_code": "string"})
    company_df["stock_code"] = (
        company_df["stock_code"]
        .astype("string")
        .str.extract(r"(\d+)", expand=False)
        .str.zfill(6)
    )

    if "company_name" not in company_df.columns:
        return pd.DataFrame(columns=["stock_code", "company_name"])

    return (
        company_df[["stock_code", "company_name"]]
        .dropna()
        .drop_duplicates("stock_code")
    )


def build_firm_year_corpus(raw_xml_dir: Path) -> pd.DataFrame:
    rows = []

    xml_files = sorted(raw_xml_dir.glob("*.xml"))
    if not xml_files:
        raise FileNotFoundError(f"No XML files found in {raw_xml_dir}")

    for xml_path in xml_files:
        stock_code, fiscal_year, rcept_no = parse_xml_filename(xml_path)
        xml_text = xml_path.read_text(encoding="utf-8", errors="ignore")
        sections = extract_target_sections(xml_text)

        document = " ".join(section["text"] for section in sections)
        document_norm = normalize_report_text(document)

        rows.append({
            "stock_code": stock_code,
            "fiscal_year": fiscal_year,
            "rcept_no": rcept_no,
            "file_name": xml_path.name,
            "document": document,
            "document_norm": document_norm,
            "section_count": len({section["section"] for section in sections}),
            "total_word_count": len(document_norm.split()),
            "total_char_count": len(document_norm),
            "esg_year": fiscal_year + 1,
        })

    corpus_df = pd.DataFrame(rows)
    company_df = load_company_names(COMPANY_MASTER_PATH)

    corpus_df = corpus_df.merge(company_df, on="stock_code", how="left")

    ordered_cols = [
        "stock_code",
        "company_name",
        "fiscal_year",
        "rcept_no",
        "file_name",
        "document",
        "document_norm",
        "section_count",
        "total_word_count",
        "total_char_count",
        "esg_year",
    ]

    return (
        corpus_df[ordered_cols]
        .sort_values(["stock_code", "fiscal_year", "rcept_no"])
        .reset_index(drop=True)
    )


corpus_df = build_firm_year_corpus(RAW_XML_DIR)

print("rows:", len(corpus_df))
print("section_count distribution:")
print(corpus_df["section_count"].value_counts().sort_index())
display(corpus_df[["stock_code", "company_name", "fiscal_year", "esg_year", "section_count", "total_word_count", "total_char_count"]].head())

rows: 381
section_count distribution:
section_count
3    381
Name: count, dtype: int64


,stock_code,company_name,fiscal_year,esg_year,section_count,total_word_count,total_char_count
0,000020,동화약품,2022,2023,3,7863,38149
1,000020,동화약품,2023,2024,3,8120,38663
2,000020,동화약품,2024,2025,3,8152,39455
3,000040,KR모터스,2022,2023,3,4256,20332
4,000040,KR모터스,2023,2024,3,4943,22598


## EDA

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

GRADE_ORDER = ["D", "C", "B", "B+", "A", "A+"]
GRADE_MAP = {"D": 0, "C": 1, "B": 2, "B+": 3, "A": 4, "A+": 5}

eda_df = corpus_df.copy()

if "company_master" not in globals():
    company_master_path = Path("/content/drive/MyDrive/company_master.csv")
    company_master = pd.read_csv(company_master_path, dtype={"stock_code": "string"}, encoding="utf-8-sig")

for df in [eda_df, company_master]:
    df["stock_code"] = df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)
    df["fiscal_year"] = pd.to_numeric(df["fiscal_year"], errors="coerce").astype("Int64")

needed_master_cols = [
    col for col in [
        "stock_code", "fiscal_year", "industry",
        "esg_grade", "e_grade", "s_grade", "g_grade"
    ]
    if col in company_master.columns
]

merge_cols = [
    col for col in ["industry", "esg_grade", "e_grade", "s_grade", "g_grade"]
    if col not in eda_df.columns and col in company_master.columns
]

if merge_cols:
    eda_df = eda_df.merge(
        company_master[["stock_code", "fiscal_year"] + merge_cols]
        .drop_duplicates(["stock_code", "fiscal_year"]),
        on=["stock_code", "fiscal_year"],
        how="left",
    )

if "esg_grade_num" not in eda_df.columns and "esg_grade" in eda_df.columns:
    eda_df["esg_grade_num"] = eda_df["esg_grade"].map(GRADE_MAP)

print("1. Overall corpus summary")
overall_summary = pd.DataFrame([{
    "firm_years": len(eda_df),
    "firms": eda_df["stock_code"].nunique(),
    "fiscal_years": eda_df["fiscal_year"].nunique(),
    "industries": eda_df["industry"].nunique() if "industry" in eda_df.columns else np.nan,
    "missing_esg_grade": eda_df["esg_grade"].isna().sum() if "esg_grade" in eda_df.columns else np.nan,
}])
display(overall_summary)

print("2. Yearly sample distribution")
year_sample = (
    eda_df
    .groupby("fiscal_year")
    .agg(
        firm_years=("stock_code", "size"),
        firms=("stock_code", "nunique"),
    )
    .reset_index()
)
display(year_sample)

if "industry" in eda_df.columns:
    print("3. Industry distribution")
    industry_sample = (
        eda_df
        .groupby("industry", dropna=False)
        .agg(
            firm_years=("stock_code", "size"),
            firms=("stock_code", "nunique"),
        )
        .assign(share=lambda df: df["firm_years"] / df["firm_years"].sum())
        .sort_values("firm_years", ascending=False)
        .reset_index()
    )
    display(industry_sample)

if "esg_grade_num" in eda_df.columns:
    print("4. Yearly ESG grade statistics")
    year_grade_stats = (
        eda_df
        .groupby("fiscal_year")["esg_grade_num"]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .reset_index()
    )
    display(year_grade_stats)

if "esg_grade" in eda_df.columns:
    print("5. Yearly ESG grade distribution")
    year_grade_dist = (
        eda_df
        .pivot_table(
            index="fiscal_year",
            columns="esg_grade",
            values="stock_code",
            aggfunc="count",
            fill_value=0,
        )
        .reindex(columns=GRADE_ORDER, fill_value=0)
    )
    display(year_grade_dist)

    print("6. Overall ESG grade distribution")
    overall_grade_dist = (
        eda_df["esg_grade"]
        .value_counts(dropna=False)
        .reindex(GRADE_ORDER)
        .rename_axis("esg_grade")
        .reset_index(name="firm_years")
    )
    overall_grade_dist["share"] = overall_grade_dist["firm_years"] / overall_grade_dist["firm_years"].sum()
    display(overall_grade_dist)

if "industry" in eda_df.columns and "esg_grade_num" in eda_df.columns:
    print("7. Industry-level ESG grade statistics")
    industry_grade_stats = (
        eda_df
        .groupby("industry", dropna=False)["esg_grade_num"]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .sort_values("count", ascending=False)
        .reset_index()
    )
    display(industry_grade_stats)

length_cols = [col for col in ["total_word_count", "total_char_count", "kiwi_term_count"] if col in eda_df.columns]
if length_cols:
    print("8. Document length statistics")
    length_stats = (
        eda_df[length_cols]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .T
        .reset_index()
        .rename(columns={"index": "variable"})
    )
    display(length_stats)

    print("9. Yearly document length statistics")
    yearly_length_stats = (
        eda_df
        .groupby("fiscal_year")[length_cols]
        .agg(["count", "mean", "median", "std", "min", "max"])
    )
    display(yearly_length_stats)

1. Overall corpus summary


,firm_years,firms,fiscal_years,industries,missing_esg_grade
0,381,127,3,22,0


2. Yearly sample distribution


,fiscal_year,firm_years,firms
0,2022,127,127
1,2023,127,127
2,2024,127,127


3. Industry distribution


...

,industry,count,mean,median,std,min,max
0,NaN,297,2.309764,3.0,1.608152,0,5
1,금융지주,15,4.333333,4.0,0.487950,4,5
2,인터넷/플랫폼,6,4.166667,4.0,0.408248,4,5
3,화학,6,3.333333,3.0,0.516398,3,4
4,게임,3,4.000000,4.0,0.000000,4,4
5,건설,3,0.000000,0.0,0.000000,0,0
6,IT서비스,3,4.000000,4.0,0.000000,4,4
7,바이오의약품,3,3.333333,4.0,1.154701,2,4
8,미디어/콘텐츠,3,3.333333,3.0,0.577350,3,4
9,물류,3,3.333333,4.0,1.154701,2,4


8. Document length statistics


,variable,count,mean,median,std,min,max
0,total_word_count,381.0,14974.167979,11141.0,12295.430087,1829.0,71775.0
1,total_char_count,381.0,72929.984252,54910.0,61153.186683,8853.0,360969.0


9. Yearly document length statistics


total_word_count                                             \
                       count          mean   median           std   min   
fiscal_year                                                               
2022                     127  14597.141732  11261.0  11943.647912  1829   
2023                     127  15151.535433  11022.0  12534.639008  1854   
2024                     127  15173.826772  11331.0  12487.969457  1899   

                   total_char_count                                       \
               max            count          mean   median           std   
fiscal_year                                                                
2022         64219              127  71219.275591  54885.0  59352.295611   
2023         66051              127  73619.338583  55290.0  62049.343556   
2024         71775              127  73951.338583  54544.0  62459.595188   

                           
              min     max  
fiscal_year                
2022         8853  321492  
2023         9036  329885  
2024         9129  360969

## 증분설명력 분석

In [4]:
import subprocess
import sys
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer

try:
    from kiwipiepy import Kiwi
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kiwipiepy"])
    from kiwipiepy import Kiwi

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer

SEED_DICTIONARY_PATH = Path("/content/drive/MyDrive/seed_dictionary.csv")

MODEL_NAME = "dragonkue/BGE-m3-ko"
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
BATCH_SIZE = 64 if DEVICE == "cuda" else 16

MIN_TERM_FREQ = 3
MIN_DOC_FREQ = 2
MAX_CANDIDATES = 30000
THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.80, 0.90, 1.00]

GRADE_MAP = {"D": 0, "C": 1, "B": 2, "B+": 3, "A": 4, "A+": 5}

def normalize_term(value):
    text = "" if pd.isna(value) else str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text.strip(" \t\r\n\"'`.,;:()[]{}<>")


def token_key(term):
    return normalize_term(term).replace(" ", "_")


def threshold_label(theta):
    return f"{float(theta):.2f}".replace(".", "_")


def is_good_term(term):
    term = normalize_term(term)
    if len(term) < 2 or term.isdigit():
        return False
    if re.fullmatch(r"[0-9.,%/?????]+", term):
        return False
    if re.search(r"[\u3131-\u318E]", term):
        return False
    return True


def split_seed_terms(row):
    values = [row.get("seed_term", "")]
    pattern = row.get("pattern", "")
    if pd.notna(pattern):
        values.extend(str(pattern).split("|"))

    terms = []
    seen = set()
    for value in values:
        term = normalize_term(value)
        if is_good_term(term) and term.lower() != "nan" and term not in seen:
            terms.append(term)
            seen.add(term)
    return terms


def build_seed_query_df(seed_df):
    rows = []
    for idx, row in seed_df.reset_index(drop=True).iterrows():
        dimension = normalize_term(row.get("dimension", ""))
        if dimension not in {"E", "S", "G"}:
            continue

        seed_term = normalize_term(row.get("seed_term", ""))
        for query_term in split_seed_terms(row):
            rows.append({
                "seed_id": idx,
                "dimension": dimension,
                "seed_term": seed_term,
                "query_term": query_term,
                "query_text": query_term,
            })

    return (
        pd.DataFrame(rows)
        .drop_duplicates(["dimension", "seed_term", "query_term"])
        .reset_index(drop=True)
    )


seed_source_df = pd.read_csv(SEED_DICTIONARY_PATH, encoding="utf-8-sig")
seed_query_df = build_seed_query_df(seed_source_df)
seed_terms = set(seed_query_df["query_term"])

print("Seed query terms")
display(seed_query_df.groupby("dimension").size().rename("query_terms").reset_index())

kiwi = Kiwi()
NOUN_TAGS = {"NNG", "NNP", "SL"}


def kiwi_noun_candidates(text):
    candidates = []
    current = []

    for token in kiwi.tokenize("" if pd.isna(text) else str(text)):
        form = normalize_term(token.form)
        if token.tag in NOUN_TAGS and is_good_term(form):
            candidates.append(form)
            current.append(form)
        else:
            if len(current) >= 2:
                phrase = normalize_term(" ".join(current))
                if is_good_term(phrase):
                    candidates.append(phrase)
                for i in range(len(current) - 1):
                    bigram = normalize_term(" ".join(current[i:i + 2]))
                    if is_good_term(bigram):
                        candidates.append(bigram)
            current = []

    if len(current) >= 2:
        phrase = normalize_term(" ".join(current))
        if is_good_term(phrase):
            candidates.append(phrase)
        for i in range(len(current) - 1):
            bigram = normalize_term(" ".join(current[i:i + 2]))
            if is_good_term(bigram):
                candidates.append(bigram)

    return candidates


appendix_embedding_doc_df = corpus_df[[
    "stock_code", "company_name", "fiscal_year", "esg_year",
    "rcept_no", "total_word_count", "document_norm",
]].copy()
appendix_embedding_doc_df["kiwi_terms"] = appendix_embedding_doc_df["document_norm"].apply(kiwi_noun_candidates)
appendix_embedding_doc_df["kiwi_document"] = appendix_embedding_doc_df["kiwi_terms"].apply(
    lambda terms: " ".join(token_key(term) for term in terms)
)
appendix_embedding_doc_df["kiwi_term_count"] = appendix_embedding_doc_df["kiwi_terms"].apply(len)

term_counter = Counter()
doc_counter = Counter()
for terms in appendix_embedding_doc_df["kiwi_terms"]:
    term_counter.update(terms)
    doc_counter.update(set(terms))

candidate_rows = []
for term, freq in term_counter.most_common():
    if term in seed_terms:
        continue
    doc_freq = doc_counter[term]
    if freq < MIN_TERM_FREQ or doc_freq < MIN_DOC_FREQ:
        continue
    candidate_rows.append({
        "candidate_term": term,
        "term_frequency": freq,
        "doc_frequency": doc_freq,
    })

candidate_df = pd.DataFrame(candidate_rows).head(MAX_CANDIDATES).reset_index(drop=True)
if candidate_df.empty:
    raise ValueError("No candidate terms left. Lower MIN_TERM_FREQ/MIN_DOC_FREQ or inspect Kiwi extraction.")

print("candidate terms:", len(candidate_df))
display(candidate_df.head(20))


def encode_texts(model, texts, batch_size):
    embeddings = model.encode(
        list(texts),
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    return np.asarray(embeddings, dtype=np.float32)


model = SentenceTransformer(MODEL_NAME, device=DEVICE)
seed_embeddings = encode_texts(model, seed_query_df["query_text"], BATCH_SIZE)
candidate_embeddings = encode_texts(model, candidate_df["candidate_term"], BATCH_SIZE)
similarity_matrix = seed_embeddings @ candidate_embeddings.T

best_rows = []
for cand_i, cand in candidate_df.reset_index(drop=True).iterrows():
    sims = similarity_matrix[:, cand_i]
    best_i = int(np.argmax(sims))
    best_seed = seed_query_df.iloc[best_i]

    dim_scores = {
        dimension: float(np.max(sims[list(indices)]))
        for dimension, indices in seed_query_df.groupby("dimension").groups.items()
    }
    sorted_dim_scores = sorted(dim_scores.items(), key=lambda x: x[1], reverse=True)
    best_dim, best_dim_score = sorted_dim_scores[0]
    second_dim_score = sorted_dim_scores[1][1] if len(sorted_dim_scores) > 1 else np.nan

    best_rows.append({
        "candidate_term": cand["candidate_term"],
        "term_frequency": cand["term_frequency"],
        "doc_frequency": cand["doc_frequency"],
        "best_dimension": best_dim,
        "best_dimension_score": best_dim_score,
        "second_dimension_score": second_dim_score,
        "dimension_margin": best_dim_score - second_dim_score,
        "best_seed_term": best_seed["seed_term"],
        "best_query_term": best_seed["query_term"],
    })

scored_candidate_df = pd.DataFrame(best_rows).sort_values(
    ["best_dimension", "best_dimension_score", "dimension_margin", "candidate_term"],
    ascending=[True, False, False, True],
).reset_index(drop=True)

print("Top embedding-expanded candidates")
display(scored_candidate_df.head(30))

vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    min_df=1,
    norm=None,
)
appendix_embedding_tfidf_matrix = vectorizer.fit_transform(appendix_embedding_doc_df["kiwi_document"])
appendix_embedding_vocab = vectorizer.vocabulary_


def prepare_terms(terms):
    prepared = []
    seen = set()
    for term in terms:
        term = normalize_term(term)
        key = token_key(term)
        if term and key in appendix_embedding_vocab and term not in seen:
            prepared.append((term, appendix_embedding_vocab[key]))
            seen.add(term)
    return prepared


def add_embedding_scores(score_df, meta_rows, dictionary, threshold, dimension, terms, prefix):
    prepared_terms = prepare_terms(terms)
    term_set = {term for term, _ in prepared_terms}
    tfidf_cols = [col for _, col in prepared_terms]

    count_feature = f"{dimension}_{prefix}_count"
    tfidf_feature = f"{dimension}_{prefix}_tfidf"

    score_df[count_feature] = [
        sum(1 for term in doc_terms if term in term_set)
        for doc_terms in appendix_embedding_doc_df["kiwi_terms"]
    ]
    score_df[tfidf_feature] = (
        np.asarray(appendix_embedding_tfidf_matrix[:, tfidf_cols].sum(axis=1)).ravel()
        if tfidf_cols else np.zeros(len(score_df), dtype=float)
    )

    meta_rows.append({
        "dictionary": dictionary,
        "threshold": threshold,
        "dimension": dimension,
        "prefix": prefix,
        "term_count": len(prepared_terms),
        "dictionary_term_count": len({normalize_term(term) for term in terms if normalize_term(term)}),
    })


appendix_embedding_score_df = appendix_embedding_doc_df[[
    "stock_code", "company_name", "fiscal_year", "esg_year",
    "rcept_no", "total_word_count", "kiwi_term_count",
]].copy()
appendix_embedding_meta_rows = []

for dimension in ["E", "S", "G"]:
    seed_terms_dim = seed_query_df.loc[seed_query_df["dimension"].eq(dimension), "query_term"].tolist()
    add_embedding_scores(appendix_embedding_score_df, appendix_embedding_meta_rows, "kiwi_seed_only", np.nan, dimension, seed_terms_dim, "kiwi_seed")

for theta in THRESHOLDS:
    kept = scored_candidate_df[scored_candidate_df["best_dimension_score"].ge(theta)]
    prefix = f"kiwi_expanded_{threshold_label(theta)}"
    for dimension in ["E", "S", "G"]:
        terms = kept.loc[kept["best_dimension"].eq(dimension), "candidate_term"].tolist()
        add_embedding_scores(appendix_embedding_score_df, appendix_embedding_meta_rows, "kiwi_embedding_expanded", theta, dimension, terms, prefix)

appendix_embedding_meta_df = pd.DataFrame(appendix_embedding_meta_rows)
for prefix in appendix_embedding_meta_df["prefix"].drop_duplicates():
    for score_type in ["count", "tfidf"]:
        appendix_embedding_score_df[f"ESG_{prefix}_{score_type}"] = sum(
            appendix_embedding_score_df[f"{dimension}_{prefix}_{score_type}"]
            for dimension in ["E", "S", "G"]
        )

company_master = company_master.copy()
company_master["stock_code"] = company_master["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)
appendix_embedding_score_df["stock_code"] = appendix_embedding_score_df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)

for col in ["fiscal_year", "esg_year"]:
    company_master[col] = pd.to_numeric(company_master[col], errors="coerce").astype("Int64")
    appendix_embedding_score_df[col] = pd.to_numeric(appendix_embedding_score_df[col], errors="coerce").astype("Int64")

for col in ["esg_grade", "e_grade", "s_grade", "g_grade"]:
    company_master[f"{col}_num"] = company_master[col].map(GRADE_MAP)

grade_cols = [
    "stock_code", "fiscal_year", "esg_year",
    "esg_grade_num", "e_grade_num", "s_grade_num", "g_grade_num",
]
appendix_embedding_analysis_df = appendix_embedding_score_df.merge(
    company_master[grade_cols].drop_duplicates(["stock_code", "fiscal_year", "esg_year"]),
    on=["stock_code", "fiscal_year", "esg_year"],
    how="left",
)

print("Kiwi embedding expanded term counts by threshold")
display(
    appendix_embedding_meta_df[appendix_embedding_meta_df["dictionary"].eq("kiwi_embedding_expanded")]
    .pivot_table(index="threshold", columns="dimension", values="term_count", aggfunc="first")
)

# Incremental explanatory power: does Kiwi embedding expanded-only score improve over seed?
import statsmodels.api as sm


def appendix_embedding_zscore(series):
    series = series.astype(float)
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return series * 0
    return (series - series.mean()) / std


def appendix_embedding_fit_ols(data, y_col, x_cols):
    reg_df = data[[y_col] + x_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(reg_df) < len(x_cols) + 3:
        return None, None, reg_df
    y = reg_df[y_col].astype(float)
    X = reg_df[x_cols].apply(appendix_embedding_zscore)
    X = sm.add_constant(X, has_constant="add")
    plain_model = sm.OLS(y, X).fit()
    hc3_model = sm.OLS(y, X).fit(cov_type="HC3")
    return plain_model, hc3_model, reg_df


embedding_incremental_rows = []
seed_prefix = "kiwi_seed"
for _, group in appendix_embedding_meta_df[appendix_embedding_meta_df["dictionary"].eq("kiwi_embedding_expanded")].groupby(["threshold", "prefix"], dropna=False):
    threshold = group["threshold"].iloc[0]
    prefix = group["prefix"].iloc[0]
    added_term_count = int(group["term_count"].sum())
    added_dictionary_term_count = int(group["dictionary_term_count"].sum())

    for score_type in ["count", "tfidf"]:
        seed_feature = f"ESG_{seed_prefix}_{score_type}"
        added_feature = f"ESG_{prefix}_{score_type}"
        if seed_feature not in appendix_embedding_analysis_df.columns or added_feature not in appendix_embedding_analysis_df.columns:
            continue

        # Kiwi embedding expansion candidates exclude seed terms, so the expanded feature is already added-only.
        base_plain, base_hc3, base_df = appendix_embedding_fit_ols(
            appendix_embedding_analysis_df,
            "esg_grade_num",
            [seed_feature, "total_word_count"],
        )
        full_plain, full_hc3, full_df = appendix_embedding_fit_ols(
            appendix_embedding_analysis_df,
            "esg_grade_num",
            [seed_feature, added_feature, "total_word_count"],
        )
        if base_plain is None or full_plain is None:
            continue

        f_stat, f_pvalue, df_diff = full_plain.compare_f_test(base_plain)
        embedding_incremental_rows.append({
            "dictionary": "kiwi_embedding_expanded",
            "threshold": threshold,
            "score_type": score_type,
            "seed_feature": seed_feature,
            "added_feature": added_feature,
            "n": int(full_plain.nobs),
            "added_term_count": added_term_count,
            "added_dictionary_term_count": added_dictionary_term_count,
            "base_r2": float(base_plain.rsquared),
            "full_r2": float(full_plain.rsquared),
            "delta_r2": float(full_plain.rsquared - base_plain.rsquared),
            "base_adj_r2": float(base_plain.rsquared_adj),
            "full_adj_r2": float(full_plain.rsquared_adj),
            "delta_adj_r2": float(full_plain.rsquared_adj - base_plain.rsquared_adj),
            "base_aic": float(base_plain.aic),
            "full_aic": float(full_plain.aic),
            "delta_aic_full_minus_base": float(full_plain.aic - base_plain.aic),
            "base_bic": float(base_plain.bic),
            "full_bic": float(full_plain.bic),
            "delta_bic_full_minus_base": float(full_plain.bic - base_plain.bic),
            "partial_f": float(f_stat),
            "partial_f_p_value": float(f_pvalue),
            "df_diff": float(df_diff),
            "added_coef_hc3": float(full_hc3.params[added_feature]),
            "added_p_value_hc3": float(full_hc3.pvalues[added_feature]),
        })

appendix_embedding_incremental_ols_df = (
    pd.DataFrame(embedding_incremental_rows)
    .sort_values(["score_type", "delta_adj_r2"], ascending=[True, False])
    .reset_index(drop=True)
)

print("Kiwi embedding added-only incremental OLS over seed baseline")
display(appendix_embedding_incremental_ols_df)

if not appendix_embedding_incremental_ols_df.empty:
    print("Best Kiwi embedding threshold by adjusted R2 gain")
    display(
        appendix_embedding_incremental_ols_df
        .sort_values("delta_adj_r2", ascending=False)
        .head(10)
    )



Seed query terms


,dimension,query_terms
0,E,18
1,G,18
2,S,18


candidate terms: 30000


,candidate_term,term_frequency,doc_frequency
0,찬성,81083,365
1,찬성 찬성,61577,346
2,사업,41360,381
3,이사,38865,381
4,자산,36471,381
5,회사,33717,381
6,사항,29949,381
7,시장,28884,381
8,금융,28652,381
9,관리,28546,381


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


...

Kiwi embedding added-only incremental OLS over seed baseline


/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:2284: RuntimeWarning: invalid value encountered in scalar divide
  f_value = (ssr_restr - ssr_full) / df_diff / ssr_full * df_full
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:2284: RuntimeWarning: invalid value encountered in scalar divide
  f_value = (ssr_restr - ssr_full) / df_diff / ssr_full * df_full


,dictionary,threshold,score_type,seed_feature,added_feature,n,added_term_count,added_dictionary_term_count,base_r2,full_r2,...,full_aic,delta_aic_full_minus_base,base_bic,full_bic,delta_bic_full_minus_base,partial_f,partial_f_p_value,df_diff,added_coef_hc3,added_p_value_hc3
0,kiwi_embedding_expanded,0.80,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_80_count,381,117,117,0.300475,0.431880,...,1242.047157,-77.274079,1331.149634,1257.818354,-73.331280,87.199138,8.676306e-19,1.0,1.045435,1.329356e-18
1,kiwi_embedding_expanded,0.70,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_70_count,381,436,436,0.300475,0.398758,...,1263.636258,-55.684978,1331.149634,1279.407455,-51.742178,61.626971,4.347795e-14,1.0,1.410815,4.599282e-13
2,kiwi_embedding_expanded,0.65,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_65_count,381,702,702,0.300475,0.383353,...,1273.274991,-46.046245,1331.149634,1289.046188,-42.103445,50.669549,5.558292e-12,1.0,1.267070,1.531716e-10
3,kiwi_embedding_expanded,0.55,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_55_count,381,2040,2040,0.300475,0.359067,...,1287.992291,-31.328944,1331.149634,1303.763489,-27.386145,34.464494,9.526062e-09,1.0,2.131318,2.152197e-08
4,kiwi_embedding_expanded,0.60,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_60_count,381,1103,1103,0.300475,0.356447,...,1289.546518,-29.774717,1331.149634,1305.317716,-25.831918,32.789411,2.100087e-08,1.0,1.727175,4.411904e-08
5,kiwi_embedding_expanded,0.45,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_45_count,381,9804,9804,0.300475,0.354490,...,1290.703439,-28.617797,1331.149634,1306.474636,-24.674997,31.546958,3.785274e-08,1.0,4.302926,1.315387e-09
6,kiwi_embedding_expanded,0.40,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_40_count,381,19423,19423,0.300475,0.351569,...,1292.423544,-26.897692,1331.149634,1308.194741,-22.954892,29.706643,9.099700e-08,1.0,4.478180,2.420102e-08
7,kiwi_embedding_expanded,0.50,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_50_count,381,4381,4381,0.300475,0.351026,...,1292.742818,-26.578417,1331.149634,1308.514016,-22.635618,29.365969,1.071040e-07,1.0,2.882795,1.951747e-07
8,kiwi_embedding_expanded,0.30,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_30_count,381,29678,29678,0.300475,0.338660,...,1299.934548,-19.386687,1331.149634,1315.705746,-15.443888,21.767374,4.282086e-06,1.0,3.988139,9.687908e-07
9,kiwi_embedding_expanded,0.10,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_10_count,381,30000,30000,0.300475,0.338448,...,1300.056220,-19.265015,1331.149634,1315.827418,-15.322216,21.640048,4.559391e-06,1.0,3.932704,1.043905e-06


Best Kiwi embedding threshold by adjusted R2 gain


,dictionary,threshold,score_type,seed_feature,added_feature,n,added_term_count,added_dictionary_term_count,base_r2,full_r2,...,full_aic,delta_aic_full_minus_base,base_bic,full_bic,delta_bic_full_minus_base,partial_f,partial_f_p_value,df_diff,added_coef_hc3,added_p_value_hc3
0,kiwi_embedding_expanded,0.80,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_80_count,381,117,117,0.300475,0.431880,...,1242.047157,-77.274079,1331.149634,1257.818354,-73.331280,87.199138,8.676306e-19,1.0,1.045435,1.329356e-18
13,kiwi_embedding_expanded,0.80,tfidf,ESG_kiwi_seed_tfidf,ESG_kiwi_expanded_0_80_tfidf,381,117,117,0.278901,0.398455,...,1263.828179,-67.065798,1342.722374,1279.599376,-63.122998,74.926812,1.439184e-16,1.0,1.289646,2.245726e-17
1,kiwi_embedding_expanded,0.70,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_70_count,381,436,436,0.300475,0.398758,...,1263.636258,-55.684978,1331.149634,1279.407455,-51.742178,61.626971,4.347795e-14,1.0,1.410815,4.599282e-13
2,kiwi_embedding_expanded,0.65,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_65_count,381,702,702,0.300475,0.383353,...,1273.274991,-46.046245,1331.149634,1289.046188,-42.103445,50.669549,5.558292e-12,1.0,1.267070,1.531716e-10
3,kiwi_embedding_expanded,0.55,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_55_count,381,2040,2040,0.300475,0.359067,...,1287.992291,-31.328944,1331.149634,1303.763489,-27.386145,34.464494,9.526062e-09,1.0,2.131318,2.152197e-08
4,kiwi_embedding_expanded,0.60,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_60_count,381,1103,1103,0.300475,0.356447,...,1289.546518,-29.774717,1331.149634,1305.317716,-25.831918,32.789411,2.100087e-08,1.0,1.727175,4.411904e-08
14,kiwi_embedding_expanded,0.65,tfidf,ESG_kiwi_seed_tfidf,ESG_kiwi_expanded_0_65_tfidf,381,702,702,0.278901,0.334064,...,1302.572668,-28.321308,1342.722374,1318.343865,-24.378509,31.229156,4.402621e-08,1.0,1.346885,6.074339e-09
5,kiwi_embedding_expanded,0.45,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_45_count,381,9804,9804,0.300475,0.354490,...,1290.703439,-28.617797,1331.149634,1306.474636,-24.674997,31.546958,3.785274e-08,1.0,4.302926,1.315387e-09
6,kiwi_embedding_expanded,0.40,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_40_count,381,19423,19423,0.300475,0.351569,...,1292.423544,-26.897692,1331.149634,1308.194741,-22.954892,29.706643,9.099700e-08,1.0,4.478180,2.420102e-08
7,kiwi_embedding_expanded,0.50,count,ESG_kiwi_seed_count,ESG_kiwi_expanded_0_50_count,381,4381,4381,0.300475,0.351026,...,1292.742818,-26.578417,1331.149634,1308.514016,-22.635618,29.365969,1.071040e-07,1.0,2.882795,1.951747e-07


## 상관분석

In [5]:
from scipy.stats import spearmanr

FINAL_EXPANDED_DICTIONARY_PATH = Path("/content/drive/MyDrive/candidate_kiwi_embedding_threshold_0_80_dictionary.csv")

GRADE_BY_DIMENSION = {
    "E": "e_grade_num",
    "S": "s_grade_num",
    "G": "g_grade_num",
    "ESG": "esg_grade_num",
}


def load_expanded_dictionary(path):
    expanded_df = pd.read_csv(path, encoding="utf-8-sig")
    term_col = next(
        col for col in ["candidate_term", "query_term", "seed_term", "term"]
        if col in expanded_df.columns
    )
    loaded_df = expanded_df[["dimension", term_col]].rename(columns={term_col: "candidate_term"}).copy()
    loaded_df["dictionary"] = "final_expanded"
    loaded_df["candidate_term"] = loaded_df["candidate_term"].map(normalize_term)
    loaded_df = loaded_df[loaded_df["dimension"].isin(["E", "S", "G"])]
    loaded_df = loaded_df[loaded_df["candidate_term"].ne("")]
    return loaded_df.drop_duplicates(["dictionary", "dimension", "candidate_term"]).reset_index(drop=True)


def safe_spearman(x, y):
    data = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(data) < 3 or data["x"].nunique() < 2 or data["y"].nunique() < 2:
        return np.nan, np.nan, len(data)

    rho, pvalue = spearmanr(data["x"], data["y"])
    return float(rho), float(pvalue), len(data)


required_previous_vars = [
    "appendix_embedding_doc_df",
    "appendix_embedding_tfidf_matrix",
    "appendix_embedding_vocab",
    "company_master",
]
missing_vars = [name for name in required_previous_vars if name not in globals()]
if missing_vars:
    raise RuntimeError(f"Run the ????? ?? cell first. Missing: {missing_vars}")

if not str(FINAL_EXPANDED_DICTIONARY_PATH).strip():
    raise ValueError("FINAL_EXPANDED_DICTIONARY_PATH must point to the selected threshold 0.80 expanded dictionary.")

correlation_dictionary_df = load_expanded_dictionary(FINAL_EXPANDED_DICTIONARY_PATH)

print("Dictionary term counts")
display(
    correlation_dictionary_df
    .groupby(["dictionary", "dimension"])
    .size()
    .rename("dictionary_term_count")
    .reset_index()
)


def prepare_final_dictionary_terms(terms):
    prepared = []
    seen = set()
    for term in terms:
        term = normalize_term(term)
        key = token_key(term)
        if term and key in appendix_embedding_vocab and term not in seen:
            prepared.append((term, key, appendix_embedding_vocab[key]))
            seen.add(term)
    return prepared


def add_final_dimension_scores(score_df, feature_meta_rows, dictionary, dimension, terms):
    prepared_terms = prepare_final_dictionary_terms(terms)
    term_set = {term for term, _, _ in prepared_terms}
    tfidf_cols = [col for _, _, col in prepared_terms]

    count_feature = f"{dimension}_{dictionary}_count"
    tfidf_feature = f"{dimension}_{dictionary}_tfidf"

    score_df[count_feature] = [
        sum(1 for term in doc_terms if term in term_set)
        for doc_terms in appendix_embedding_doc_df["kiwi_terms"]
    ]
    score_df[tfidf_feature] = (
        np.asarray(appendix_embedding_tfidf_matrix[:, tfidf_cols].sum(axis=1)).ravel()
        if tfidf_cols else np.zeros(len(score_df), dtype=float)
    )

    feature_meta_rows.append({
        "dictionary": dictionary,
        "dimension": dimension,
        "term_count": len(prepared_terms),
        "dictionary_term_count": len({normalize_term(term) for term in terms if normalize_term(term)}),
    })


final_score_df = appendix_embedding_doc_df[[
    "stock_code", "company_name", "fiscal_year", "esg_year",
    "rcept_no", "total_word_count", "kiwi_term_count",
]].copy()
final_feature_meta_rows = []

for (dictionary, dimension), group in correlation_dictionary_df.groupby(["dictionary", "dimension"]):
    add_final_dimension_scores(final_score_df, final_feature_meta_rows, dictionary, dimension, group["candidate_term"].tolist())

for dictionary in correlation_dictionary_df["dictionary"].drop_duplicates():
    for score_type in ["count", "tfidf"]:
        final_score_df[f"ESG_{dictionary}_{score_type}"] = sum(
            final_score_df[f"{dimension}_{dictionary}_{score_type}"]
            for dimension in ["E", "S", "G"]
        )

final_feature_meta_df = pd.DataFrame(final_feature_meta_rows)
final_score_df["stock_code"] = final_score_df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)

for col in ["fiscal_year", "esg_year"]:
    final_score_df[col] = pd.to_numeric(final_score_df[col], errors="coerce").astype("Int64")

final_grade_cols = [
    "stock_code", "fiscal_year", "esg_year",
    "esg_grade_num", "e_grade_num", "s_grade_num", "g_grade_num",
]

final_level_analysis_df = final_score_df.merge(
    company_master[final_grade_cols].drop_duplicates(["stock_code", "fiscal_year", "esg_year"]),
    on=["stock_code", "fiscal_year", "esg_year"],
    how="left",
)

final_corr_rows = []
for dictionary in correlation_dictionary_df["dictionary"].drop_duplicates():
    dictionary_meta_df = final_feature_meta_df[final_feature_meta_df["dictionary"].eq(dictionary)]
    for dimension in ["E", "S", "G", "ESG"]:
        grade_col = GRADE_BY_DIMENSION[dimension]
        term_count = (
            int(dictionary_meta_df.loc[dictionary_meta_df["dimension"].eq(dimension), "term_count"].iloc[0])
            if dimension != "ESG" else int(dictionary_meta_df["term_count"].sum())
        )
        dictionary_term_count = (
            int(dictionary_meta_df.loc[dictionary_meta_df["dimension"].eq(dimension), "dictionary_term_count"].iloc[0])
            if dimension != "ESG" else int(dictionary_meta_df["dictionary_term_count"].sum())
        )

        for score_type in ["count", "tfidf"]:
            feature = f"{dimension}_{dictionary}_{score_type}"
            rho, pvalue, n = safe_spearman(final_level_analysis_df[feature], final_level_analysis_df[grade_col])
            final_corr_rows.append({
                "scope": "pooled",
                "dictionary": dictionary,
                "dimension": dimension,
                "score_type": score_type,
                "feature": feature,
                "grade_col": grade_col,
                "term_count": term_count,
                "dictionary_term_count": dictionary_term_count,
                "spearman_rho": rho,
                "p_value": pvalue,
                "n": n,
            })

final_correlation_df = pd.DataFrame(final_corr_rows)

final_year_corr_rows = []
for _, row in final_correlation_df.iterrows():
    for fiscal_year, year_df in final_level_analysis_df.groupby("fiscal_year"):
        rho, pvalue, n = safe_spearman(year_df[row["feature"]], year_df[row["grade_col"]])
        final_year_corr_rows.append({
            "scope": "yearly",
            "fiscal_year": fiscal_year,
            "dictionary": row["dictionary"],
            "dimension": row["dimension"],
            "score_type": row["score_type"],
            "feature": row["feature"],
            "grade_col": row["grade_col"],
            "term_count": row["term_count"],
            "dictionary_term_count": row["dictionary_term_count"],
            "spearman_rho": rho,
            "p_value": pvalue,
            "n": n,
        })

final_year_correlation_df = pd.DataFrame(final_year_corr_rows)

print("Pooled Spearman: final expanded dictionary threshold 0.80")
display(final_correlation_df.sort_values(["dictionary", "dimension", "score_type"]).reset_index(drop=True))

print("Pooled pivot")
display(final_correlation_df.pivot_table(
    index=["dictionary", "score_type"],
    columns="dimension",
    values="spearman_rho",
    aggfunc="first",
))

print("Yearly Spearman: final expanded dictionary threshold 0.80")
display(final_year_correlation_df.sort_values(["fiscal_year", "dictionary", "dimension", "score_type"]).reset_index(drop=True))

print("Yearly pivot")
display(final_year_correlation_df.pivot_table(
    index=["dictionary", "score_type", "fiscal_year"],
    columns="dimension",
    values="spearman_rho",
    aggfunc="first",
))


Dictionary term counts


,dictionary,dimension,dictionary_term_count
0,final_expanded,E,61
1,final_expanded,G,53
2,final_expanded,S,38


Pooled Spearman: final expanded dictionary threshold 0.80


,scope,dictionary,dimension,score_type,feature,grade_col,term_count,dictionary_term_count,spearman_rho,p_value,n
0,pooled,final_expanded,E,count,E_final_expanded_count,e_grade_num,53,61,0.509890,1.325744e-26,381
1,pooled,final_expanded,E,tfidf,E_final_expanded_tfidf,e_grade_num,53,61,0.507881,2.245186e-26,381
2,pooled,final_expanded,ESG,count,ESG_final_expanded_count,esg_grade_num,130,152,0.725383,1.902341e-63,381
3,pooled,final_expanded,ESG,tfidf,ESG_final_expanded_tfidf,esg_grade_num,130,152,0.722463,1.028750e-62,381
4,pooled,final_expanded,G,count,G_final_expanded_count,g_grade_num,45,53,0.638388,5.304510e-45,381
5,pooled,final_expanded,G,tfidf,G_final_expanded_tfidf,g_grade_num,45,53,0.650512,3.286872e-47,381
6,pooled,final_expanded,S,count,S_final_expanded_count,s_grade_num,32,38,0.628203,3.192998e-43,381
7,pooled,final_expanded,S,tfidf,S_final_expanded_tfidf,s_grade_num,32,38,0.630800,1.139741e-43,381


Pooled pivot


dimension                         E       ESG         G         S
dictionary     score_type                                        
final_expanded count       0.509890  0.725383  0.638388  0.628203
               tfidf       0.507881  0.722463  0.650512  0.630800

Yearly Spearman: final expanded dictionary threshold 0.80


,scope,fiscal_year,dictionary,dimension,score_type,feature,grade_col,term_count,dictionary_term_count,spearman_rho,p_value,n
0,yearly,2022,final_expanded,E,count,E_final_expanded_count,e_grade_num,53,61,0.539848,5.768849e-11,127
1,yearly,2022,final_expanded,E,tfidf,E_final_expanded_tfidf,e_grade_num,53,61,0.538722,6.433145e-11,127
2,yearly,2022,final_expanded,ESG,count,ESG_final_expanded_count,esg_grade_num,130,152,0.749935,3.484736e-24,127
3,yearly,2022,final_expanded,ESG,tfidf,ESG_final_expanded_tfidf,esg_grade_num,130,152,0.750645,2.989214e-24,127
4,yearly,2022,final_expanded,G,count,G_final_expanded_count,g_grade_num,45,53,0.675143,3.185810e-18,127
5,yearly,2022,final_expanded,G,tfidf,G_final_expanded_tfidf,g_grade_num,45,53,0.680516,1.361943e-18,127
6,yearly,2022,final_expanded,S,count,S_final_expanded_count,s_grade_num,32,38,0.705560,2.016937e-20,127
7,yearly,2022,final_expanded,S,tfidf,S_final_expanded_tfidf,s_grade_num,32,38,0.709750,9.545423e-21,127
8,yearly,2023,final_expanded,E,count,E_final_expanded_count,e_grade_num,53,61,0.534480,9.666043e-11,127
9,yearly,2023,final_expanded,E,tfidf,E_final_expanded_tfidf,e_grade_num,53,61,0.532902,1.122972e-10,127


Yearly pivot


dimension                                     E       ESG         G         S
dictionary     score_type fiscal_year                                        
final_expanded count      2022         0.539848  0.749935  0.675143  0.705560
                          2023         0.534480  0.717330  0.618758  0.609969
                          2024         0.447561  0.708779  0.624316  0.560536
               tfidf      2022         0.538722  0.750645  0.680516  0.709750
                          2023         0.532902  0.715277  0.638288  0.604887
                          2024         0.447957  0.699579  0.642944  0.570864

In [6]:
import subprocess
import sys

from scipy.stats import kruskal, mannwhitneyu

try:
    import scikit_posthocs as sp
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-posthocs"])
    import scikit_posthocs as sp

HIGH_GRADE_MIN = 4  # A, A+, S
LOW_GRADE_MAX = 3   # D, C, B, B+


def run_mannwhitney_feature_test(data, feature_meta, high_min=HIGH_GRADE_MIN, low_max=LOW_GRADE_MAX):
    rows = []
    for _, row in feature_meta.iterrows():
        feature = row["feature"]
        grade_col = row["grade_col"]
        tmp = (
            data[[feature, grade_col]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .copy()
        )

        high_values = tmp.loc[tmp[grade_col] >= high_min, feature]
        low_values = tmp.loc[tmp[grade_col] <= low_max, feature]

        statistic, pvalue = np.nan, np.nan
        if len(high_values) >= 2 and len(low_values) >= 2:
            try:
                statistic, pvalue = mannwhitneyu(high_values, low_values, alternative="two-sided")
            except ValueError:
                pass

        rank_biserial = (
            2 * statistic / (len(high_values) * len(low_values)) - 1
            if pd.notna(statistic) and len(high_values) > 0 and len(low_values) > 0
            else np.nan
        )

        rows.append({
            "dictionary": row["dictionary"],
            "dimension": row["dimension"],
            "score_type": row["score_type"],
            "feature": feature,
            "grade_col": grade_col,
            "test": "Mann-Whitney U",
            "group_rule": "high=A 이상, low=B+ 이하",
            "n_high": len(high_values),
            "n_low": len(low_values),
            "high_mean": high_values.mean(),
            "low_mean": low_values.mean(),
            "mean_diff_high_minus_low": high_values.mean() - low_values.mean(),
            "high_median": high_values.median(),
            "low_median": low_values.median(),
            "mannwhitney_u": statistic,
            "rank_biserial": rank_biserial,
            "p_value": pvalue,
        })

    return pd.DataFrame(rows)


def run_kruskal_feature_test(data, feature_meta):
    rows = []
    for _, row in feature_meta.iterrows():
        feature = row["feature"]
        grade_col = row["grade_col"]
        tmp = (
            data[[feature, grade_col]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .copy()
        )

        grouped = [
            (grade, group[feature].dropna())
            for grade, group in tmp.groupby(grade_col)
        ]
        grouped = [(grade, values) for grade, values in grouped if len(values) > 0]
        groups = [values.values for _, values in grouped]

        statistic, pvalue = np.nan, np.nan
        if len(groups) >= 2 and sum(len(values) for values in groups) >= 3:
            try:
                statistic, pvalue = kruskal(*groups)
            except ValueError:
                pass

        rows.append({
            "dictionary": row["dictionary"],
            "dimension": row["dimension"],
            "score_type": row["score_type"],
            "feature": feature,
            "grade_col": grade_col,
            "test": "Kruskal-Wallis",
            "grade_groups": ", ".join(str(grade) for grade, _ in grouped),
            "grade_group_counts": ", ".join(f"{grade}:{len(values)}" for grade, values in grouped),
            "n_groups": len(groups),
            "n_total": sum(len(values) for values in groups),
            "kruskal_h": statistic,
            "p_value": pvalue,
        })

    return pd.DataFrame(rows)


def run_dunn_posthoc_test(data, feature_meta, p_adjust="holm"):
    rows = []
    for _, row in feature_meta.iterrows():
        feature = row["feature"]
        grade_col = row["grade_col"]
        tmp = (
            data[[feature, grade_col]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .copy()
        )

        grade_counts = tmp[grade_col].value_counts().sort_index()
        if len(grade_counts) < 2:
            continue

        try:
            posthoc = sp.posthoc_dunn(tmp, val_col=feature, group_col=grade_col, p_adjust=p_adjust)
        except ValueError:
            continue

        grades = list(posthoc.index)
        for i, grade_a in enumerate(grades):
            for grade_b in grades[i + 1:]:
                values_a = tmp.loc[tmp[grade_col].eq(grade_a), feature]
                values_b = tmp.loc[tmp[grade_col].eq(grade_b), feature]
                rows.append({
                    "dictionary": row["dictionary"],
                    "dimension": row["dimension"],
                    "score_type": row["score_type"],
                    "feature": feature,
                    "grade_col": grade_col,
                    "test": "Dunn posthoc",
                    "p_adjust": p_adjust,
                    "grade_a": grade_a,
                    "grade_b": grade_b,
                    "n_a": len(values_a),
                    "n_b": len(values_b),
                    "median_a": values_a.median(),
                    "median_b": values_b.median(),
                    "median_diff_a_minus_b": values_a.median() - values_b.median(),
                    "p_value_adj": posthoc.loc[grade_a, grade_b],
                })

    return pd.DataFrame(rows)


final_mannwhitney_df = run_mannwhitney_feature_test(final_level_analysis_df, final_correlation_df)
final_kruskal_df = run_kruskal_feature_test(final_level_analysis_df, final_correlation_df)
final_dunn_df = run_dunn_posthoc_test(final_level_analysis_df, final_correlation_df)

print("Mann-Whitney U test: high grade (A 이상) vs low grade (B+ 이하)")
display(final_mannwhitney_df.sort_values(["dictionary", "dimension", "score_type"]).reset_index(drop=True))

print("Mann-Whitney pivot: p-value")
display(final_mannwhitney_df.pivot_table(
    index=["dictionary", "score_type"],
    columns="dimension",
    values="p_value",
    aggfunc="first",
))

print("Kruskal-Wallis test: all observed grade groups")
display(final_kruskal_df.sort_values(["dictionary", "dimension", "score_type"]).reset_index(drop=True))

print("Kruskal-Wallis pivot: p-value")
display(final_kruskal_df.pivot_table(
    index=["dictionary", "score_type"],
    columns="dimension",
    values="p_value",
    aggfunc="first",
))

print("Dunn posthoc test after Kruskal-Wallis: Holm-adjusted p-value")
display(final_dunn_df.sort_values(["dictionary", "dimension", "score_type", "p_value_adj"]).reset_index(drop=True))

print("Dunn posthoc significant pairs: adjusted p < 0.05")
display(
    final_dunn_df[final_dunn_df["p_value_adj"] < 0.05]
    .sort_values(["dictionary", "dimension", "score_type", "p_value_adj"])
    .reset_index(drop=True)
)


Mann-Whitney U test: high grade (A 이상) vs low grade (B+ 이하)


,dictionary,dimension,score_type,feature,grade_col,test,group_rule,n_high,n_low,high_mean,low_mean,mean_diff_high_minus_low,high_median,low_median,mannwhitney_u,rank_biserial,p_value
0,final_expanded,E,count,E_final_expanded_count,e_grade_num,Mann-Whitney U,"high=A 이상, low=B+ 이하",172,209,69.383721,21.277512,48.106209,24.500000,3.000000,27217.5,0.514271,3.961334e-18
1,final_expanded,E,tfidf,E_final_expanded_tfidf,e_grade_num,Mann-Whitney U,"high=A 이상, low=B+ 이하",172,209,141.567031,42.664363,98.902668,49.838758,6.136789,27215.0,0.514132,4.116702e-18
2,final_expanded,ESG,count,ESG_final_expanded_count,esg_grade_num,Mann-Whitney U,"high=A 이상, low=B+ 이하",152,229,543.947368,278.253275,265.694093,459.500000,247.000000,29743.0,0.708975,9.885776e-32
3,final_expanded,ESG,tfidf,ESG_final_expanded_tfidf,esg_grade_num,Mann-Whitney U,"high=A 이상, low=B+ 이하",152,229,680.924758,328.035831,352.888927,570.530401,278.779025,29593.0,0.700356,5.267902e-31
4,final_expanded,G,count,G_final_expanded_count,g_grade_num,Mann-Whitney U,"high=A 이상, low=B+ 이하",103,278,412.844660,240.176259,172.668401,352.000000,222.000000,23443.0,0.637424,1.197404e-21
5,final_expanded,G,tfidf,G_final_expanded_tfidf,g_grade_num,Mann-Whitney U,"high=A 이상, low=B+ 이하",103,278,453.483948,257.392619,196.091329,395.025780,232.189239,23749.0,0.658797,5.150861e-23
6,final_expanded,S,count,S_final_expanded_count,s_grade_num,Mann-Whitney U,"high=A 이상, low=B+ 이하",233,148,67.218884,34.222973,32.995911,57.000000,23.000000,27853.0,0.615416,4.115866e-24
7,final_expanded,S,tfidf,S_final_expanded_tfidf,s_grade_num,Mann-Whitney U,"high=A 이상, low=B+ 이하",233,148,89.190653,42.628178,46.562474,69.843726,27.462641,27925.0,0.619592,2.069282e-24


Mann-Whitney pivot: p-value


dimension                             E           ESG             G  \
dictionary     score_type                                             
final_expanded count       3.961334e-18  9.885776e-32  1.197404e-21   
               tfidf       4.116702e-18  5.267902e-31  5.150861e-23   

dimension                             S  
dictionary     score_type                
final_expanded count       4.115866e-24  
               tfidf       2.069282e-24

Kruskal-Wallis test: all observed grade groups


...

dimension                             E           ESG             G  \
dictionary     score_type                                             
final_expanded count       8.424391e-22  2.621864e-42  2.479964e-32   
               tfidf       1.372164e-21  3.756247e-42  1.534416e-33   

dimension                             S  
dictionary     score_type                
final_expanded count       1.153668e-35  
               tfidf       1.051433e-35

Dunn posthoc test after Kruskal-Wallis: Holm-adjusted p-value


,dictionary,dimension,score_type,feature,grade_col,test,p_adjust,grade_a,grade_b,n_a,n_b,median_a,median_b,median_diff_a_minus_b,p_value_adj
0,final_expanded,E,count,E_final_expanded_count,e_grade_num,Dunn posthoc,holm,0,4,42,130,0.000000,23.000000,-23.000000,1.277760e-15
1,final_expanded,E,count,E_final_expanded_count,e_grade_num,Dunn posthoc,holm,1,4,64,130,2.000000,23.000000,-21.000000,2.621431e-11
2,final_expanded,E,count,E_final_expanded_count,e_grade_num,Dunn posthoc,holm,0,5,42,42,0.000000,26.500000,-26.500000,4.410440e-10
3,final_expanded,E,count,E_final_expanded_count,e_grade_num,Dunn posthoc,holm,0,3,42,74,0.000000,12.500000,-12.500000,5.068464e-07
4,final_expanded,E,count,E_final_expanded_count,e_grade_num,Dunn posthoc,holm,1,5,64,42,2.000000,26.500000,-24.500000,1.580178e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,final_expanded,S,tfidf,S_final_expanded_tfidf,s_grade_num,Dunn posthoc,holm,3,5,33,136,50.904056,86.069723,-35.165667,4.641394e-01
116,final_expanded,S,tfidf,S_final_expanded_tfidf,s_grade_num,Dunn posthoc,holm,2,3,14,33,46.050507,50.904056,-4.853549,5.805769e-01
117,final_expanded,S,tfidf,S_final_expanded_tfidf,s_grade_num,Dunn posthoc,holm,0,1,67,34,22.687957,24.701709,-2.013752,7.100956e-01
118,final_expanded,S,tfidf,S_final_expanded_tfidf,s_grade_num,Dunn posthoc,holm,2,4,14,97,46.050507,50.800995,-4.750488,7.100956e-01


Dunn posthoc significant pairs: adjusted p < 0.05


,dictionary,dimension,score_type,feature,grade_col,test,p_adjust,grade_a,grade_b,n_a,n_b,median_a,median_b,median_diff_a_minus_b,p_value_adj
0,final_expanded,E,count,E_final_expanded_count,e_grade_num,Dunn posthoc,holm,0,4,42,130,0.000000,23.000000,-23.000000,1.277760e-15
1,final_expanded,E,count,E_final_expanded_count,e_grade_num,Dunn posthoc,holm,1,4,64,130,2.000000,23.000000,-21.000000,2.621431e-11
2,final_expanded,E,count,E_final_expanded_count,e_grade_num,Dunn posthoc,holm,0,5,42,42,0.000000,26.500000,-26.500000,4.410440e-10
3,final_expanded,E,count,E_final_expanded_count,e_grade_num,Dunn posthoc,holm,0,3,42,74,0.000000,12.500000,-12.500000,5.068464e-07
4,final_expanded,E,count,E_final_expanded_count,e_grade_num,Dunn posthoc,holm,1,5,64,42,2.000000,26.500000,-24.500000,1.580178e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,final_expanded,S,tfidf,S_final_expanded_tfidf,s_grade_num,Dunn posthoc,holm,1,3,34,33,24.701709,50.904056,-26.202347,4.614539e-06
83,final_expanded,S,tfidf,S_final_expanded_tfidf,s_grade_num,Dunn posthoc,holm,1,4,34,97,24.701709,50.800995,-26.099285,5.513711e-06
84,final_expanded,S,tfidf,S_final_expanded_tfidf,s_grade_num,Dunn posthoc,holm,4,5,97,136,50.800995,86.069723,-35.268728,1.938597e-04
85,final_expanded,S,tfidf,S_final_expanded_tfidf,s_grade_num,Dunn posthoc,holm,0,2,67,14,22.687957,46.050507,-23.362550,7.079581e-03


## ESG 감성분석-1

In [7]:
# 등급 수준 분석용 ESG 문장 평균 감성지표
# 기존 변화량 분석 변수(v_2_sentiment_analysis_df)는 덮어쓰지 않고 level_* 변수로 별도 저장합니다.

if "pd" not in globals():
    import pandas as pd
if "np" not in globals():
    import numpy as np
if "Path" not in globals():
    from pathlib import Path
if "display" not in globals():
    from IPython.display import display
if "spearmanr" not in globals():
    from scipy.stats import spearmanr
if "kruskal" not in globals():
    from scipy.stats import kruskal
if "sm" not in globals():
    import statsmodels.api as sm
if "Kiwi" not in globals():
    from kiwipiepy import Kiwi

if "sentiment_pipe" not in globals():
    if "pipeline" not in globals():
        from transformers import pipeline
    sentiment_pipe = pipeline(
        "sentiment-analysis",
        model="snunlp/KR-FinBert-SC",
        tokenizer="snunlp/KR-FinBert-SC",
        truncation=True,
        max_length=256,
    )

if "kiwi" not in globals():
    kiwi = Kiwi()

if "seed_words" in globals():
    level_seed_words = {str(term).strip() for term in seed_words if str(term).strip()}
else:
    level_dictionary_path = globals().get(
        "FINAL_EXPANDED_DICTIONARY_PATH",
        Path("/content/drive/MyDrive/candidate_kiwi_embedding_threshold_0_80_dictionary.csv"),
    )
    level_dictionary_df = pd.read_csv(level_dictionary_path, encoding="utf-8-sig")
    level_seed_col = "seed_term" if "seed_term" in level_dictionary_df.columns else "candidate_term"
    level_seed_words = {
        str(term).strip()
        for term in level_dictionary_df[level_seed_col].dropna()
        if str(term).strip()
    }

def parse_level_sentiment_output(result):
    label_raw = str(result["label"])
    label = label_raw.upper().replace(" ", "_")
    score = float(result["score"])

    if label.startswith("LABEL_"):
        try:
            label_id = int(label.split("_", 1)[1])
            label = str(sentiment_pipe.model.config.id2label.get(label_id, label)).upper().replace(" ", "_")
        except ValueError:
            pass

    if any(token in label for token in ["NEG", "NEGATIVE", "부정", "하락", "악재"]):
        return "negative", -score
    if any(token in label for token in ["POS", "POSITIVE", "긍정", "상승", "호재"]):
        return "positive", score
    if any(token in label for token in ["NEU", "NEUTRAL", "중립"]):
        return "neutral", 0.0
    return label_raw.lower(), np.nan

level_sentence_rows = []
level_doc_rows = []

for _, row in corpus_df.iterrows():
    text = str(row.get("document_norm", row.get("document", "")))
    if not text.strip():
        continue

    sentences = kiwi.split_into_sents(text)
    level_total_words = 0

    for sent_idx, sent in enumerate(sentences):
        sent_text = sent.text
        tokens = [
            token.form
            for token in kiwi.tokenize(sent_text)
            if token.tag.startswith("N") or token.tag.startswith("V")
        ]
        level_total_words += len(tokens)
        found_seed_terms = [token for token in tokens if token in level_seed_words]
        matched_terms = sorted(set(found_seed_terms))
        if not matched_terms:
            continue

        level_sentence_rows.append({
            "stock_code": row["stock_code"],
            "company_name": row.get("company_name", ""),
            "fiscal_year": int(row["fiscal_year"]),
            "esg_year": int(row["fiscal_year"]) + 1,
            "sentence_idx": sent_idx,
            "sentence": sent_text,
            "matched_seed_terms": "; ".join(matched_terms),
            "matched_seed_term_count": len(found_seed_terms),
            "total_word_count": row.get("total_word_count", np.nan),
        })

    level_doc_rows.append({
        "stock_code": row["stock_code"],
        "company_name": row.get("company_name", ""),
        "fiscal_year": int(row["fiscal_year"]),
        "esg_year": int(row["fiscal_year"]) + 1,
        "esg_sentence_count": len(sentences),
        "total_word_count": level_total_words,
    })

level_sentence_df = pd.DataFrame(level_sentence_rows)
level_doc_stats_df = pd.DataFrame(level_doc_rows)
print("level seed sentence rows:", len(level_sentence_df))

if level_sentence_df.empty:
    raise ValueError("ESG seed 문장이 추출되지 않았습니다. seed 사전 또는 corpus_df를 확인하세요.")

level_batch_size = int(globals().get("BATCH_SIZE", 32))
level_raw_results = sentiment_pipe(
    level_sentence_df["sentence"].fillna("").tolist(),
    batch_size=level_batch_size,
)
level_parsed = [parse_level_sentiment_output(result) for result in level_raw_results]

level_sentence_df["sentiment_label"] = [label for label, _ in level_parsed]
level_sentence_df["sentiment_score"] = [score for _, score in level_parsed]
level_sentence_df["positive_sentiment"] = (level_sentence_df["sentiment_label"] == "positive").astype(int)
level_sentence_df["negative_sentiment"] = (level_sentence_df["sentiment_label"] == "negative").astype(int)
level_sentence_df["neutral_sentiment"] = (level_sentence_df["sentiment_label"] == "neutral").astype(int)

display(level_sentence_df["sentiment_label"].value_counts(dropna=False).rename("sentence_count"))

level_sentiment_agg = (
    level_sentence_df
    .groupby(["stock_code", "company_name", "fiscal_year", "esg_year"], as_index=False)
    .agg(
        level_esg_sentence_count=("sentence", "count"),
        level_positive_esg_sentence_count=("positive_sentiment", "sum"),
        level_negative_esg_sentence_count=("negative_sentiment", "sum"),
        level_neutral_esg_sentence_count=("neutral_sentiment", "sum"),
        level_mean_esg_sentiment=("sentiment_score", "mean"),
    )
)

level_sentiment_agg["level_positive_esg_share"] = (
    level_sentiment_agg["level_positive_esg_sentence_count"]
    / level_sentiment_agg["level_esg_sentence_count"].replace(0, np.nan)
)
level_sentiment_agg["level_negative_esg_share"] = (
    level_sentiment_agg["level_negative_esg_sentence_count"]
    / level_sentiment_agg["level_esg_sentence_count"].replace(0, np.nan)
)
level_sentiment_agg["level_neutral_esg_share"] = (
    level_sentiment_agg["level_neutral_esg_sentence_count"]
    / level_sentiment_agg["level_esg_sentence_count"].replace(0, np.nan)
)

level_base_cols = ["stock_code", "company_name", "fiscal_year"]
if "total_word_count" in corpus_df.columns:
    level_base_cols.append("total_word_count")
level_base_panel = corpus_df[level_base_cols].copy()
level_base_panel["esg_year"] = level_base_panel["fiscal_year"].astype(int) + 1

level_sentiment_analysis_df = level_base_panel.merge(
    level_sentiment_agg.drop(columns=["company_name"]),
    on=["stock_code", "fiscal_year", "esg_year"],
    how="left",
)

level_count_cols = [
    "level_esg_sentence_count",
    "level_positive_esg_sentence_count",
    "level_negative_esg_sentence_count",
    "level_neutral_esg_sentence_count",
]
level_sentiment_analysis_df[level_count_cols] = level_sentiment_analysis_df[level_count_cols].fillna(0)
level_sentiment_analysis_df["level_mean_esg_sentiment"] = level_sentiment_analysis_df["level_mean_esg_sentiment"].fillna(0)

if "company_master" not in globals():
    company_master_path = globals().get("COMPANY_MASTER_PATH", Path("/content/drive/MyDrive/company_master.csv"))
    company_master = pd.read_csv(company_master_path, dtype={"stock_code": "string"}, encoding="utf-8-sig")

level_grade_map = globals().get("GRADE_MAP", {"D": 0, "C": 1, "B": 2, "B+": 3, "A": 4, "A+": 5})
level_grade_cols = [
    col for col in ["stock_code", "fiscal_year", "industry", "esg_grade", "e_grade", "s_grade", "g_grade"]
    if col in company_master.columns
]
level_grade_df = company_master[level_grade_cols].copy()
level_grade_df["stock_code"] = level_grade_df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)
level_grade_df["fiscal_year"] = pd.to_numeric(level_grade_df["fiscal_year"], errors="coerce").astype("Int64")
level_grade_df["esg_year"] = level_grade_df["fiscal_year"] + 1
if "esg_grade_num" not in level_grade_df.columns and "esg_grade" in level_grade_df.columns:
    level_grade_df["esg_grade_num"] = level_grade_df["esg_grade"].map(level_grade_map)

level_grade_analysis_df = level_sentiment_analysis_df.copy()
level_grade_analysis_df["stock_code"] = level_grade_analysis_df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)
level_grade_analysis_df["fiscal_year"] = pd.to_numeric(level_grade_analysis_df["fiscal_year"], errors="coerce").astype("Int64")
level_grade_analysis_df["esg_year"] = level_grade_analysis_df["fiscal_year"] + 1
level_grade_analysis_df = level_grade_analysis_df.merge(
    level_grade_df.drop_duplicates(["stock_code", "fiscal_year", "esg_year"]),
    on=["stock_code", "fiscal_year", "esg_year"],
    how="left",
)

print("level analysis rows:", len(level_grade_analysis_df))
print("missing esg_grade_num:", level_grade_analysis_df["esg_grade_num"].isna().sum())

level_features = [
    "level_mean_esg_sentiment",
    "level_positive_esg_share",
    "level_negative_esg_share",
    "level_neutral_esg_share",
    "level_esg_sentence_count",
    "total_word_count",
]
level_features = [feature for feature in level_features if feature in level_grade_analysis_df.columns]

level_spearman_rows = []
for feature in level_features:
    tmp = level_grade_analysis_df[["esg_grade_num", feature]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(tmp) < 3 or tmp["esg_grade_num"].nunique() < 2 or tmp[feature].nunique() < 2:
        rho, p_value = np.nan, np.nan
    else:
        rho, p_value = spearmanr(tmp[feature], tmp["esg_grade_num"])
    level_spearman_rows.append({
        "feature": feature,
        "n": len(tmp),
        "spearman_rho": rho,
        "p_value": p_value,
    })

level_spearman_df = pd.DataFrame(level_spearman_rows).sort_values("spearman_rho", ascending=False)
print("Spearman correlation with ESG grade level")
display(level_spearman_df)

def level_zscore(series):
    series = series.astype(float)
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return series * 0
    return (series - series.mean()) / std

def run_level_ols(data, y_col, x_cols):
    reg_df = data[[y_col] + x_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    y = reg_df[y_col].astype(float)
    X = reg_df[x_cols].apply(level_zscore)
    X = sm.add_constant(X, has_constant="add")
    model = sm.OLS(y, X).fit(cov_type="HC3")
    result = pd.DataFrame({
        "term": model.params.index,
        "coef": model.params.values,
        "std_err_HC3": model.bse.values,
        "t": model.tvalues.values,
        "p_value": model.pvalues.values,
        "r2": model.rsquared,
        "n": int(model.nobs),
    })
    return model, result

level_model_specs = {
    "L1_mean_sentiment_only": ["level_mean_esg_sentiment"],
    "L2_mean_sentiment_controls": ["level_mean_esg_sentiment", "level_esg_sentence_count", "total_word_count"],
    "L3_pos_neg_share_controls": ["level_positive_esg_share", "level_negative_esg_share", "level_esg_sentence_count", "total_word_count"],
}

level_ols_rows = []
for model_name, x_cols in level_model_specs.items():
    x_cols = [col for col in x_cols if col in level_grade_analysis_df.columns]
    model, result = run_level_ols(level_grade_analysis_df, "esg_grade_num", x_cols)
    print("\n" + model_name, "n=", int(model.nobs), "R2=", round(model.rsquared, 4))
    display(result)
    result = result.assign(model=model_name)
    level_ols_rows.append(result)

level_ols_df = pd.concat(level_ols_rows, ignore_index=True)

level_group_df = level_grade_analysis_df.replace([np.inf, -np.inf], np.nan).dropna(
    subset=["esg_grade_num", "level_positive_esg_share", "level_negative_esg_share"]
).copy()

if not level_group_df.empty:
    level_pos_cut = level_group_df["level_positive_esg_share"].median()
    level_neg_cut = level_group_df["level_negative_esg_share"].median()
    level_group_df["level_pos_neg_group"] = np.where(
        level_group_df["level_positive_esg_share"] >= level_pos_cut,
        "pos_high",
        "pos_low",
    ) + "__" + np.where(
        level_group_df["level_negative_esg_share"] >= level_neg_cut,
        "neg_high",
        "neg_low",
    )

    level_group_summary_df = (
        level_group_df.groupby("level_pos_neg_group")
        .agg(
            n=("stock_code", "size"),
            mean_esg_grade=("esg_grade_num", "mean"),
            median_esg_grade=("esg_grade_num", "median"),
            mean_positive_share=("level_positive_esg_share", "mean"),
            mean_negative_share=("level_negative_esg_share", "mean"),
        )
        .reset_index()
        .sort_values("mean_esg_grade", ascending=False)
    )
    print("Positive/negative share high-low groups")
    display(level_group_summary_df)

    level_groups = [
        group["esg_grade_num"].dropna().values
        for _, group in level_group_df.groupby("level_pos_neg_group")
    ]
    level_kw_stat, level_kw_p = kruskal(*level_groups)
    level_kruskal_df = pd.DataFrame([{
        "test": "Kruskal-Wallis",
        "group_col": "level_pos_neg_group",
        "statistic": level_kw_stat,
        "p_value": level_kw_p,
    }])
    display(level_kruskal_df)


config.json:   0%|          | 0.00/881 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/406M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: snunlp/KR-FinBert-SC
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

...


L3_pos_neg_share_controls n= 381 R2= 0.271


,term,coef,std_err_HC3,t,p_value,r2,n
0,const,2.598425,0.071867,36.155985,2.996092e-286,0.27099,381
1,level_positive_esg_share,0.296302,0.084516,3.505853,4.551469e-04,0.27099,381
2,level_negative_esg_share,-0.023055,0.069410,-0.332161,7.397678e-01,0.27099,381
3,level_esg_sentence_count,0.070317,0.098036,0.717253,4.732180e-01,0.27099,381
4,total_word_count,0.625075,0.076442,8.177143,2.906525e-16,0.27099,381


Positive/negative share high-low groups


,level_pos_neg_group,n,mean_esg_grade,median_esg_grade,mean_positive_share,mean_negative_share
0,pos_high__neg_high,112,3.133929,3.5,0.158302,0.060247
1,pos_high__neg_low,79,3.126582,4.0,0.147834,0.015090
3,pos_low__neg_low,110,2.063636,2.0,0.032481,0.013374
2,pos_low__neg_high,80,2.062500,2.0,0.033280,0.061162


,test,group_col,statistic,p_value
0,Kruskal-Wallis,level_pos_neg_group,41.575218,4.937491e-09


## ESG 감성분석_2

In [8]:
# ESG 감성분석_2: 변화량 분석 핵심 출력
# - ESG 감성분석-1에서 만든 level_sentence_df / level_doc_stats_df를 재사용합니다.
# - 중간 head()와 보조표 출력은 줄이고, 최종 해석에 필요한 통계표만 출력합니다.

if "pd" not in globals():
    import pandas as pd
if "np" not in globals():
    import numpy as np
if "Path" not in globals():
    from pathlib import Path
if "display" not in globals():
    from IPython.display import display
if "spearmanr" not in globals():
    from scipy.stats import spearmanr
if "kruskal" not in globals():
    from scipy.stats import kruskal
if "sm" not in globals():
    import statsmodels.api as sm

required_level_vars = ["level_sentence_df", "level_doc_stats_df"]
missing_level_vars = [name for name in required_level_vars if name not in globals()]
if missing_level_vars:
    raise RuntimeError(f"먼저 'ESG 감성분석-1' 셀을 실행하세요. Missing: {missing_level_vars}")

# 1. Build firm-year sentiment metrics from sentence-level shares
sentence_source_df = level_sentence_df.copy()
sentence_source_df["positive_seed_sentence_flag"] = sentence_source_df["sentiment_label"].eq("positive").astype(int)
sentence_source_df["negative_seed_sentence_flag"] = sentence_source_df["sentiment_label"].eq("negative").astype(int)

sentiment_counts_df = (
    sentence_source_df.groupby(["stock_code", "fiscal_year"], as_index=False)
    .agg(
        s_pos_s=("positive_seed_sentence_flag", "sum"),
        s_neg_s=("negative_seed_sentence_flag", "sum"),
    )
)

v_2_sentiment_analysis_df = level_doc_stats_df.merge(
    sentiment_counts_df,
    on=["stock_code", "fiscal_year"],
    how="left",
)
for col in ["s_pos_s", "s_neg_s"]:
    v_2_sentiment_analysis_df[col] = v_2_sentiment_analysis_df[col].fillna(0)

# 2. 등급 병합: 공통 GRADE_MAP 사용
if "company_master" not in globals():
    company_master_path = globals().get("COMPANY_MASTER_PATH", Path("/content/drive/MyDrive/company_master.csv"))
    company_master = pd.read_csv(company_master_path, dtype={"stock_code": "string"}, encoding="utf-8-sig")

GRADE_MAP = globals().get("GRADE_MAP", {"D": 0, "C": 1, "B": 2, "B+": 3, "A": 4, "A+": 5})
grade_df = company_master[["stock_code", "fiscal_year", "company_name", "esg_grade"]].copy()
grade_df["stock_code"] = grade_df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)
grade_df["fiscal_year"] = pd.to_numeric(grade_df["fiscal_year"], errors="coerce").astype("Int64")
grade_df["esg_grade_num"] = grade_df["esg_grade"].map(GRADE_MAP)

v_2_sentiment_analysis_df["stock_code"] = v_2_sentiment_analysis_df["stock_code"].astype("string").str.extract(r"(\d+)",
expand=False).str.zfill(6)
v_2_sentiment_analysis_df["fiscal_year"] = pd.to_numeric(v_2_sentiment_analysis_df["fiscal_year"],
errors="coerce").astype("Int64")
v_2_sentiment_analysis_df = v_2_sentiment_analysis_df.drop(columns=["company_name", "esg_grade", "esg_grade_num"],
errors="ignore")
v_2_sentiment_analysis_df = v_2_sentiment_analysis_df.merge(
    grade_df,
    on=["stock_code", "fiscal_year"],
    how="left",
)

esg_sentence_total = v_2_sentiment_analysis_df["esg_sentence_count"].replace(0, np.nan)
v_2_sentiment_analysis_df["dictionary_label"] = "seed"
v_2_sentiment_analysis_df["positive_esg_sentence_count"] = v_2_sentiment_analysis_df["s_pos_s"]
v_2_sentiment_analysis_df["negative_esg_sentence_count"] = v_2_sentiment_analysis_df["s_neg_s"]
v_2_sentiment_analysis_df["positive_esg_share"] = (v_2_sentiment_analysis_df["s_pos_s"] / esg_sentence_total).fillna(0)
v_2_sentiment_analysis_df["negative_esg_share"] = (v_2_sentiment_analysis_df["s_neg_s"] / esg_sentence_total).fillna(0)
v_2_sentiment_analysis_df["net_sentiment_share"] = (
    v_2_sentiment_analysis_df["positive_esg_share"] - v_2_sentiment_analysis_df["negative_esg_share"]
)
v_2_sentiment_analysis_df = v_2_sentiment_analysis_df.drop(columns=["s_pos_s", "s_neg_s"], errors="ignore")

# 필요하면 재사용할 수 있도록 저장하되, 저장 경로는 출력하지 않습니다.
v_2_sentiment_analysis_df.to_csv("v_2_sentiment_analysis_df.csv", index=False, encoding="utf-8-sig")

# 3. 전년 대비 변화량 패널 구성
analysis_df = v_2_sentiment_analysis_df.copy()
use_cols = [
    "dictionary_label", "stock_code", "company_name", "fiscal_year", "esg_year", "esg_grade", "esg_grade_num",
    "net_sentiment_share", "positive_esg_share", "negative_esg_share",
    "positive_esg_sentence_count", "negative_esg_sentence_count",
    "esg_sentence_count", "total_word_count",
]
panel_df = analysis_df[use_cols].copy()
for col in [c for c in use_cols if c not in {"dictionary_label", "stock_code", "company_name", "esg_grade"}]:
    panel_df[col] = pd.to_numeric(panel_df[col], errors="coerce")

panel_df = panel_df.sort_values(["dictionary_label", "stock_code", "fiscal_year"]).reset_index(drop=True)

change_base_cols = [
    "net_sentiment_share",
    "positive_esg_share",
    "negative_esg_share",
    "positive_esg_sentence_count",
    "negative_esg_sentence_count",
    "esg_sentence_count",
    "total_word_count",
    "esg_grade_num",
]
change_df = panel_df.copy()
group_keys = ["dictionary_label", "stock_code"]
for col in change_base_cols:
    change_df[f"lag_{col}"] = change_df.groupby(group_keys)[col].shift(1)
    change_df[f"delta_{col}"] = change_df[col] - change_df[f"lag_{col}"]

change_df["year_gap"] = change_df["fiscal_year"] - change_df.groupby(group_keys)["fiscal_year"].shift(1)
change_df = change_df[change_df["year_gap"] == 1].copy()
change_df["grade_change_group"] = np.select(
    [change_df["delta_esg_grade_num"] > 0, change_df["delta_esg_grade_num"] < 0],
    ["등급 상승", "등급 하락"],
    default="등급 유지",
)

# 4. 표본 및 등급 변화 그룹 요약
sample_summary_df = pd.DataFrame([{
    "firm_year_rows": len(panel_df),
    "change_rows": len(change_df),
    "firms": panel_df["stock_code"].nunique(),
    "missing_grade_rows": int(panel_df["esg_grade_num"].isna().sum()),
}])

overall_change_summary = (
    change_df.groupby(["dictionary_label", "grade_change_group"])
    .size()
    .rename("row_count")
    .reset_index()
    .sort_values(["dictionary_label", "grade_change_group"])
)

print("1. 표본 및 등급 변화 그룹")
display(sample_summary_df)
display(overall_change_summary)

# 5. Spearman: 핵심 변수만 출력
core_change_features = [
    "net_sentiment_share",
    "delta_net_sentiment_share",
    "positive_esg_share",
    "delta_positive_esg_share",
    "negative_esg_share",
    "delta_negative_esg_share",
    "positive_esg_sentence_count",
    "delta_positive_esg_sentence_count",
    "negative_esg_sentence_count",
    "delta_negative_esg_sentence_count",
    "esg_sentence_count",
    "delta_esg_sentence_count",
    "total_word_count",
    "delta_total_word_count",
]

def spearman_delta_table(data, y_col, x_cols):
    rows = []
    for dictionary_label, group in data.groupby("dictionary_label"):
        for x_col in x_cols:
            tmp = group[[y_col, x_col]].replace([np.inf, -np.inf], np.nan).dropna()
            if len(tmp) < 3 or tmp[y_col].nunique() < 2 or tmp[x_col].nunique() < 2:
                rho, pvalue = np.nan, np.nan
            else:
                rho, pvalue = spearmanr(tmp[y_col], tmp[x_col])
            rows.append({
                "dictionary_label": dictionary_label,
                "feature": x_col,
                "target": y_col,
                "n": len(tmp),
                "spearman_rho": rho,
                "p_value": pvalue,
            })
    return pd.DataFrame(rows)

spearman_change_df = spearman_delta_table(change_df, "delta_esg_grade_num", core_change_features)
spearman_core_df = spearman_change_df.assign(
    abs_rho=lambda df: df["spearman_rho"].abs()
).sort_values("abs_rho", ascending=False).drop(columns="abs_rho")

print("2. 등급 변화량과 텍스트 지표의 Spearman 순위상관")
display(spearman_core_df)

# 6. 등급 상승/유지/하락 그룹 차이: 핵심 변수만 출력
group_cols = [
    "net_sentiment_share",
    "delta_net_sentiment_share",
    "positive_esg_share",
    "delta_positive_esg_share",
    "negative_esg_share",
    "delta_negative_esg_share",
    "positive_esg_sentence_count",
    "delta_positive_esg_sentence_count",
    "negative_esg_sentence_count",
    "delta_negative_esg_sentence_count",
    "esg_sentence_count",
    "delta_esg_sentence_count",
    "total_word_count",
    "delta_total_word_count",
]

group_summary = (
    change_df.groupby(["dictionary_label", "grade_change_group"])[group_cols]
    .agg(["count", "mean", "median"])
)

test_rows = []
for dictionary_label, dictionary_group in change_df.groupby("dictionary_label"):
    for col in group_cols:
        groups = [
            group[col].replace([np.inf, -np.inf], np.nan).dropna().values
            for _, group in dictionary_group.groupby("grade_change_group")
        ]
        groups = [group for group in groups if len(group) > 0]
        if len(groups) >= 2 and sum(len(group) for group in groups) >= 3:
            try:
                stat, pvalue = kruskal(*groups)
            except ValueError:
                stat, pvalue = np.nan, np.nan
        else:
            stat, pvalue = np.nan, np.nan
        test_rows.append({
            "dictionary_label": dictionary_label,
            "feature": col,
            "test": "Kruskal-Wallis",
            "statistic": stat,
            "p_value": pvalue,
        })

kruskal_df = pd.DataFrame(test_rows).sort_values("p_value")

print("3. 등급 변화 그룹별 차이 검정")
display(kruskal_df)

# 7. OLS + HC3: 핵심 모델만 출력
def zscore(series):
    series = series.astype(float)
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return series * 0
    return (series - series.mean()) / std

def robust_delta_ols(data, y_col, x_cols):
    reg_df = data[[y_col] + x_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    y = reg_df[y_col].astype(float)
    X = reg_df[x_cols].apply(zscore)
    X = sm.add_constant(X, has_constant="add")
    model = sm.OLS(y, X).fit(cov_type="HC3")
    result = pd.DataFrame({
        "term": model.params.index,
        "coef": model.params.values,
        "std_err_HC3": model.bse.values,
        "t": model.tvalues.values,
        "p_value": model.pvalues.values,
        "r2": model.rsquared,
        "n": int(model.nobs),
    })
    return model, result

model_specs = {
    "M1_net_sentiment_share_level": ["net_sentiment_share"],
    "M1_net_sentiment_share_delta": ["delta_net_sentiment_share"],
    "M2_net_sentiment_plus_volume_delta": ["delta_net_sentiment_share", "delta_esg_sentence_count", "delta_total_word_count"],
    "M3_pos_neg_share_plus_volume_delta": ["delta_positive_esg_share", "delta_negative_esg_share", "delta_esg_sentence_count", "delta_total_word_count"],
}

ols_rows = []
for dictionary_label, dictionary_group in change_df.groupby("dictionary_label"):
    for model_name, x_cols in model_specs.items():
        model, result = robust_delta_ols(dictionary_group, "delta_esg_grade_num", x_cols)
        result = result.assign(dictionary_label=dictionary_label, model=model_name)
        ols_rows.append(result)

ols_summary_df = pd.concat(ols_rows, ignore_index=True)
ols_core_df = ols_summary_df[ols_summary_df["term"].ne("const")].sort_values(["model", "p_value"])

print("4. 변화량 OLS 핵심 결과")
display(ols_core_df[["dictionary_label", "model", "term", "coef", "p_value", "r2", "n"]])

# 8. Core summary
net_delta_row = spearman_change_df.loc[spearman_change_df["feature"].eq("delta_net_sentiment_share")].iloc[0]
pos_delta_row = spearman_change_df.loc[spearman_change_df["feature"].eq("delta_positive_esg_share")].iloc[0]
neg_delta_row = spearman_change_df.loc[spearman_change_df["feature"].eq("delta_negative_esg_share")].iloc[0]
esg_sentence_delta_row = spearman_change_df.loc[spearman_change_df["feature"].eq("delta_esg_sentence_count")].iloc[0]
volume_delta_row = spearman_change_df.loc[spearman_change_df["feature"].eq("delta_total_word_count")].iloc[0]
print("5. Core summary")
print(
    f"delta_net_sentiment_share: "
    f"rho={net_delta_row['spearman_rho']:.3f}, p={net_delta_row['p_value']:.3f}"
)
print(
    f"delta_positive_esg_share: "
    f"rho={pos_delta_row['spearman_rho']:.3f}, p={pos_delta_row['p_value']:.3f}"
)
print(
    f"delta_negative_esg_share: "
    f"rho={neg_delta_row['spearman_rho']:.3f}, p={neg_delta_row['p_value']:.3f}"
)
print(
    f"delta_esg_sentence_count: "
    f"rho={esg_sentence_delta_row['spearman_rho']:.3f}, p={esg_sentence_delta_row['p_value']:.3f}"
)
print(
    f"delta_total_word_count: "
    f"rho={volume_delta_row['spearman_rho']:.3f}, p={volume_delta_row['p_value']:.3f}"
)


1. 표본 및 등급 변화 그룹


,firm_year_rows,change_rows,firms,missing_grade_rows
0,381,254,127,0


,dictionary_label,grade_change_group,row_count
0,seed,등급 상승,57
1,seed,등급 유지,144
2,seed,등급 하락,53


2. 등급 변화량과 텍스트 지표의 Spearman 순위상관


,dictionary_label,feature,target,n,spearman_rho,p_value
13,seed,delta_total_word_count,delta_esg_grade_num,254,0.129542,0.039105
1,seed,delta_net_sentiment_share,delta_esg_grade_num,254,0.070856,0.260548
12,seed,total_word_count,delta_esg_grade_num,254,-0.069954,0.266677
11,seed,delta_esg_sentence_count,delta_esg_grade_num,254,0.065127,0.301166
3,seed,delta_positive_esg_share,delta_esg_grade_num,254,0.052000,0.409254
7,seed,delta_positive_esg_sentence_count,delta_esg_grade_num,254,0.050368,0.424131
10,seed,esg_sentence_count,delta_esg_grade_num,254,-0.043692,0.488163
2,seed,positive_esg_share,delta_esg_grade_num,254,0.041593,0.509318
0,seed,net_sentiment_share,delta_esg_grade_num,254,0.036964,0.557603
5,seed,delta_negative_esg_share,delta_esg_grade_num,254,-0.034971,0.579054


3. 등급 변화 그룹별 차이 검정


,dictionary_label,feature,test,statistic,p_value
12,seed,total_word_count,Kruskal-Wallis,10.294552,0.005815
10,seed,esg_sentence_count,Kruskal-Wallis,7.273609,0.026336
13,seed,delta_total_word_count,Kruskal-Wallis,5.047481,0.080159
6,seed,positive_esg_sentence_count,Kruskal-Wallis,4.703369,0.095209
0,seed,net_sentiment_share,Kruskal-Wallis,3.543764,0.170013
3,seed,delta_positive_esg_share,Kruskal-Wallis,2.888830,0.235884
9,seed,delta_negative_esg_sentence_count,Kruskal-Wallis,2.618219,0.270060
5,seed,delta_negative_esg_share,Kruskal-Wallis,1.654777,0.437190
1,seed,delta_net_sentiment_share,Kruskal-Wallis,1.632676,0.442047
11,seed,delta_esg_sentence_count,Kruskal-Wallis,1.486725,0.475512


4. 변화량 OLS 핵심 결과


,dictionary_label,model,term,coef,p_value,r2,n
3,seed,M1_net_sentiment_share_delta,delta_net_sentiment_share,0.061616,0.198945,0.006486,254
1,seed,M1_net_sentiment_share_level,net_sentiment_share,0.012001,0.803305,0.000246,254
5,seed,M2_net_sentiment_plus_volume_delta,delta_net_sentiment_share,0.063753,0.226720,0.010386,254
7,seed,M2_net_sentiment_plus_volume_delta,delta_total_word_count,0.042610,0.500305,0.010386,254
6,seed,M2_net_sentiment_plus_volume_delta,delta_esg_sentence_count,0.011817,0.830618,0.010386,254
9,seed,M3_pos_neg_share_plus_volume_delta,delta_positive_esg_share,0.058544,0.321532,0.010396,254
12,seed,M3_pos_neg_share_plus_volume_delta,delta_total_word_count,0.042489,0.502217,0.010396,254
10,seed,M3_pos_neg_share_plus_volume_delta,delta_negative_esg_share,-0.026170,0.538354,0.010396,254
11,seed,M3_pos_neg_share_plus_volume_delta,delta_esg_sentence_count,0.012094,0.830562,0.010396,254


5. Core summary
delta_net_sentiment_share: rho=0.071, p=0.261
delta_positive_esg_share: rho=0.052, p=0.409
delta_negative_esg_share: rho=-0.035, p=0.579
delta_esg_sentence_count: rho=0.065, p=0.301
delta_total_word_count: rho=0.130, p=0.039


## 부록

#### 생존분석

결과에 대한 설명: p값 0.095
표본 수 70행(이벤트 11건)
표본 수가 적어서 실질적인 효과가 존재하더라도 유의확률이 높게 나옴(검정력 부족)

순수 빈도 지수는 상투적인 어미가 존재하기 때문에 문맥을 정확히 타격하지 못했다고 보여짐

위험계수는 0.0402 위험비는 1.0410
ESG 통합 지수가 1단위 증가할수록(사업보고서에 ESG 관련 긍정적인 말들을 더 많이 적을수록) 차년도 ESG 등급이 하락할 위험이 오히려 4.1%씩 증가한다는 모순적인 결과가 나옴

이러한 이유로 인해 부록에 추가함 감성분석 코드 수정 시 결과가 바뀔 수 있음!

확장 사전 수정 이후: p값과 위험계수와 위험비 전부 차이가 없어 생존분석은 해당 프로젝트에서 유의미한 값이나 변화량을 보여주지 못하여 우리가 확인하려는 부분과 먼 연관성을 보임.

In [9]:
!pip install transformers tqdm lifelines

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 14.8 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=38a5934d44b95b9c3e2ad2c03f43e07dbaa1db9ff8800cb57be8e0fd35c12e19
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma


In [10]:
#생존분석 필수 라이브러리 로드 및 가설 설정
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter

print("⏳ [생존분석 1단계] 감성분석 결과 기반 생존분석 데이터셋 준비...")

df_survival_base = v_2_sentiment_analysis_df.copy()

print(f"✅ 생존분석 기본 데이터 확보 완료: 총 {len(df_survival_base)}행 구조 확인")

⏳ [생존분석 1단계] 감성분석 결과 기반 생존분석 데이터셋 준비...
✅ 생존분석 기본 데이터 확보 완료: 총 381행 구조 확인


In [ ]:
# [감성지수 산출 로직 정상화]
print("⏳ [생존분석 2단계] 비율(Share)을 배제하고 순수 문장 개수(Count) 기반으로 지수 재연산 중...")

# 1. 꼬임 방지를 위해 순수 Count 컬럼만 명확하게 지정
pos_count_col = 'positive_esg_sentence_count'
neg_count_col = 'negative_esg_sentence_count'

# 2. 안전하게 긍정 개수에서 부정 개수를 차감하여 지수 생성 (정수형 스케일 확보)
df_survival_base['Total_ESG_idx'] = df_survival_base[pos_count_col] - df_survival_base[neg_count_col]

print("✅ 통합 ESG Net Sentiment 지수 정상화 완료 (Total_ESG_idx)")
print(df_survival_base[['stock_code', 'fiscal_year', pos_count_col, neg_count_col, 'Total_ESG_idx']].head(5))

⏳ [생존분석 2단계] 비율(Share)을 배제하고 순수 문장 개수(Count) 기반으로 지수 재연산 중...
✅ 통합 ESG Net Sentiment 지수 정상화 완료 (Total_ESG_idx)
  stock_code  fiscal_year  positive_esg_sentence_count  \
0     000020         2022                            1   
1     000020         2023                            1   
2     000020         2024                            1   
3     000040         2022                            0   
4     000040         2023                            0   

   negative_esg_sentence_count  Total_ESG_idx  
0                            2             -1  
1                            1              0  
2                            0              1  
3                            0              0  
4                            0              0  


In [12]:
# [중복 연도 제거 및 이벤트 전면 재추적]
print("⏳ [생존분석 3단계] 중복 데이터 정제 및 시계열 생존 피처(Event) 정밀 복구 중...")

# 1. 데이터프레임 내 동일 기업-동일 연도 중복 행 제거 (평균값 또는 첫 행 기준으로 단일화)
# 여기서는 데이터의 정성적 가치를 보존하기 위해 중복된 행 중 첫 번째 행만 남깁니다.
df_cleaned = df_survival_base.drop_duplicates(subset=['stock_code', 'fiscal_year'], keep='first').copy()
print(f"💡 중복 행 정제 완료: 기존 {len(df_survival_base)}행 ➡️ 정제 후 {len(df_cleaned)}행 데이터 확보")

# 2. 기업별, 연도별 시계열 엄격 정렬
df_surv = df_cleaned.sort_values(['stock_code', 'fiscal_year']).reset_index(drop=True)

# 3. 수치형 등급 피처 고정
grade_col = 'esg_grade_num'
df_surv['grade_idx_numeric'] = pd.to_numeric(df_surv[grade_col], errors='coerce')
df_surv['grade_idx_numeric'] = df_surv.groupby('stock_code')['grade_idx_numeric'].ffill()

# 4. 관측 기간(Duration) 계산
df_surv['min_year'] = df_surv.groupby('stock_code')['fiscal_year'].transform('min')
df_surv['duration'] = df_surv['fiscal_year'] - df_surv['min_year'] + 1

# 5. 시계열 무결성이 확보된 상태에서 등급 하락 이벤트(Event) 추적
df_surv['prev_grade_num'] = df_surv.groupby('stock_code')['grade_idx_numeric'].shift(1)
df_surv['event_shift'] = np.where(df_surv['grade_idx_numeric'] > df_surv['prev_grade_num'], 1, 0)

df_surv['first_grade_num'] = df_surv.groupby('stock_code')['grade_idx_numeric'].transform('first')
df_surv['event_baseline'] = np.where(df_surv['grade_idx_numeric'] > df_surv['first_grade_num'], 1, 0)

# 최종 이벤트 결합
df_surv['event'] = np.where((df_surv['event_shift'] == 1) | (df_surv['event_baseline'] == 1), 1, 0)
df_surv['event'] = df_surv['event'].fillna(0).astype(int)

print(f"🎯 피처 엔지니어링 최종 완료 ➡️ 정정된 총 등급 하락(사망 사건) 횟수: {df_surv['event'].sum()}건 발생")

⏳ [생존분석 3단계] 중복 데이터 정제 및 시계열 생존 피처(Event) 정밀 복구 중...
💡 중복 행 정제 완료: 기존 381행 ➡️ 정제 후 381행 데이터 확보
🎯 피처 엔지니어링 최종 완료 ➡️ 정정된 총 등급 하락(사망 사건) 횟수: 69건 발생


In [13]:
# [Cox PH 모델 최종 핏팅]
print("⏳ [생존분석 4단계] 정제된 70행 표본 및 11건의 순수 이벤트를 기반으로 Cox PH 모형 적합 중...")

# 분석 필수 변수만 추출 및 결측치 제거
keep_cols = ['duration', 'event', 'Total_ESG_idx']
df_model = df_surv[keep_cols].dropna()

# 콕스 비례위험모형 선언 및 학습
cph = CoxPHFitter()
cph.fit(df_model, duration_col='duration', event_col='event')

print("\n🏆 [최종 통계 결과] 미확장 Baseline 통합 ESG 지수 기반 Cox 생존분석 요약 리포트")
print("=" * 80)
cph.print_summary()
print("=" * 80)

⏳ [생존분석 4단계] 정제된 70행 표본 및 11건의 순수 이벤트를 기반으로 Cox PH 모형 적합 중...

🏆 [최종 통계 결과] 미확장 Baseline 통합 ESG 지수 기반 Cox 생존분석 요약 리포트


<lifelines.CoxPHFitter: fitted with 381 total observations, 312 right-censored observations>
             duration col = 'duration'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 381
number of events observed = 69
   partial log-likelihood = -348.55
         time fit was run = 2026-06-05 13:19:55 UTC

---
               coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                      
Total_ESG_idx  0.02      1.02      0.01            0.00            0.04                1.00                1.04

               cmp to    z    p  -log2(p)
covariate                                
Total_ESG_idx    0.00 2.09 0.04      4.76
---
Concordance = 0.59
Partial AIC = 699.09
log-likelihood ratio test = 3.56 on 1 df
-log2(p) of ll-ratio test = 4.08

In [ ]:
# [최종 지표 정형화 출력]
print("⏳ [생존분석 5단계] 보고서 삽입용 최종 핵심 수치 정형화 추출 중...")

summary_df = cph.summary
coef_val = summary_df.loc['Total_ESG_idx', 'coef']
exp_coef_val = summary_df.loc['Total_ESG_idx', 'exp(coef)']
p_val = summary_df.loc['Total_ESG_idx', 'p']
concordance_val = cph.concordance_index_
aic_val = cph.AIC_partial_

print("\n📝 [최종 보고서 본문/결론 섹션 삽입용 텍스트 요약]")
print("-" * 65)
print(f"   - 위험 계수 (Coefficient): {coef_val:.4f}")
print(f"   - 위험비 (Hazard Ratio, exp(coef)): {exp_coef_val:.4f}")
print(f"   - 유의확률 (p-value): {p_val:.4e}")
print(f"   - 모형 예측 정확도 (Concordance Index): {concordance_val:.4f}")
print(f"   - 모델 정보 손실률 (Partial AIC): {aic_val:.2f}")
print("-" * 65)
print("💡 [분석 결과 해석 가이드]")
print("   - 위험비(exp(coef))가 1보다 작고 p-value가 0.05보다 작다면,")
print("     '사업보고서 내 ESG 긍정 문장 개수가 많고 부정 문장 개수가 적을수록,")
print("      차년도 ESG 등급이 하락할 위험이 통계적으로 유의미하게 낮아진다'라고 결론을 기술합니다.")

⏳ [생존분석 5단계] 보고서 삽입용 최종 핵심 수치 정형화 추출 중...

📝 [최종 보고서 본문/결론 섹션 삽입용 텍스트 요약]
-----------------------------------------------------------------
   - 위험 계수 (Coefficient): 0.0223
   - 위험비 (Hazard Ratio, exp(coef)): 1.0225
   - 유의확률 (p-value): 3.6877e-02
   - 모형 예측 정확도 (Concordance Index): 0.5900
   - 모델 정보 손실률 (Partial AIC): 699.09
-----------------------------------------------------------------
💡 [분석 결과 해석 가이드]
   - 위험비(exp(coef))가 1보다 작고 p-value가 0.05보다 작다면,
     '사업보고서 내 ESG 긍정 문장 개수가 많고 부정 문장 개수가 적을수록,
      차년도 ESG 등급이 하락할 위험이 통계적으로 유의미하게 낮아진다'라고 결론을 기술합니다.


: 

### fasttext seed 사전 확장 상관분석/증분설명력 분석

In [3]:
import re
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer

try:
    import fasttext
    from huggingface_hub import hf_hub_download
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fasttext-wheel", "huggingface_hub"])
    import fasttext
    from huggingface_hub import hf_hub_download

try:
    from kiwipiepy import Kiwi
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kiwipiepy"])
    from kiwipiepy import Kiwi

SEED_DICTIONARY_PATH = Path("/content/drive/MyDrive/seed_dictionary.csv")
COMPANY_MASTER_PATH = Path("/content/drive/MyDrive/company_master.csv")

HF_FASTTEXT_REPO_ID = "facebook/fasttext-ko-vectors"
HF_FASTTEXT_FILENAME = "model.bin"
FASTTEXT_TOP_K = 500
THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.80, 0.90, 1.00]

GRADE_MAP = {"D": 0, "C": 1, "B": 2, "B+": 3, "A": 4, "A+": 5}
GRADE_BY_DIMENSION = {
    "E": "e_grade_num",
    "S": "s_grade_num",
    "G": "g_grade_num",
    "ESG": "esg_grade_num",
}


def normalize_term(value):
    text = "" if pd.isna(value) else str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text.strip(" \t\r\n\"'`.,;:()[]{}<>")


def token_key(term):
    return normalize_term(term).replace(" ", "_")


def threshold_label(theta):
    return f"{float(theta):.2f}".replace(".", "_")


def split_seed_terms(row):
    values = [row.get("seed_term", "")]
    pattern = row.get("pattern", "")
    if pd.notna(pattern):
        values.extend(str(pattern).split("|"))

    terms = []
    seen = set()
    for value in values:
        term = normalize_term(value)
        if term and term.lower() != "nan" and term not in seen:
            terms.append(term)
            seen.add(term)
    return terms


def build_seed_query_df(seed_df):
    rows = []
    for _, row in seed_df.iterrows():
        dimension = normalize_term(row.get("dimension", ""))
        if dimension not in {"E", "S", "G"}:
            continue

        seed_term = normalize_term(row.get("seed_term", ""))
        for query_term in split_seed_terms(row):
            rows.append({
                "dimension": dimension,
                "seed_term": seed_term,
                "query_term": query_term,
            })

    return (
        pd.DataFrame(rows)
        .drop_duplicates(["dimension", "seed_term", "query_term"])
        .reset_index(drop=True)
    )


def build_seed_dictionary_df(seed_query_df):
    return (
        seed_query_df.rename(columns={"query_term": "candidate_term"})[
            ["dimension", "seed_term", "candidate_term"]
        ]
        .assign(dictionary="seed", threshold=np.nan, source="seed", similarity=1.0)
        .drop_duplicates(["dimension", "candidate_term"])
        .reset_index(drop=True)
    )


def safe_spearman(x, y):
    data = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(data) < 3 or data["x"].nunique() < 2 or data["y"].nunique() < 2:
        return np.nan, np.nan, len(data)

    rho, pvalue = spearmanr(data["x"], data["y"])
    return float(rho), float(pvalue), len(data)


seed_source_df = pd.read_csv(SEED_DICTIONARY_PATH, encoding="utf-8-sig")
seed_query_df = build_seed_query_df(seed_source_df)
seed_dictionary_df = build_seed_dictionary_df(seed_query_df)

print("Seed query terms")
display(seed_query_df.groupby("dimension").size().rename("query_terms").reset_index())

model_path = hf_hub_download(repo_id=HF_FASTTEXT_REPO_ID, filename=HF_FASTTEXT_FILENAME)
fasttext_model = fasttext.load_model(model_path)

raw_candidate_rows = []
for _, row in seed_query_df.iterrows():
    query_term = row["query_term"]
    try:
        neighbors = fasttext_model.get_nearest_neighbors(query_term, k=FASTTEXT_TOP_K)
    except Exception as exc:
        print(f"Skipping {query_term!r}: {exc}")
        continue

    for similarity, candidate_term in neighbors:
        candidate_term = normalize_term(candidate_term)
        if not candidate_term or candidate_term == query_term:
            continue
        raw_candidate_rows.append({
            "dimension": row["dimension"],
            "seed_term": row["seed_term"],
            "query_term": query_term,
            "candidate_term": candidate_term,
            "similarity": float(similarity),
            "source": "fasttext_candidate",
        })

raw_fasttext_candidate_df = pd.DataFrame(raw_candidate_rows)
if raw_fasttext_candidate_df.empty:
    raise ValueError("No fastText candidates were generated. Check seed terms and model loading.")

raw_fasttext_candidate_df = raw_fasttext_candidate_df.drop_duplicates(
    ["dimension", "seed_term", "query_term", "candidate_term"]
).sort_values(["dimension", "seed_term", "query_term", "similarity"], ascending=[True, True, True, False])


def build_expanded_dictionary(theta):
    filtered = raw_fasttext_candidate_df[raw_fasttext_candidate_df["similarity"].ge(theta)].copy()
    expanded_rows = []
    for (dimension, candidate_term), group in filtered.groupby(["dimension", "candidate_term"], sort=False):
        best = group.sort_values("similarity", ascending=False).iloc[0]
        expanded_rows.append({
            "dictionary": f"fasttext_theta_{threshold_label(theta)}",
            "threshold": theta,
            "dimension": dimension,
            "seed_term": best["seed_term"],
            "candidate_term": candidate_term,
            "similarity": float(best["similarity"]),
            "source": "fasttext_candidate",
        })

    expanded_df = pd.DataFrame(expanded_rows)
    seed_rows = seed_dictionary_df.copy()
    seed_rows["dictionary"] = f"fasttext_theta_{threshold_label(theta)}"
    seed_rows["threshold"] = theta

    return (
        pd.concat([seed_rows, expanded_df], ignore_index=True)
        .drop_duplicates(["dimension", "candidate_term"], keep="first")
        .reset_index(drop=True)
    )


expanded_dictionary_by_threshold = {
    theta: build_expanded_dictionary(theta)
    for theta in THRESHOLDS
}

expanded_count_rows = []
for theta, dictionary_df in expanded_dictionary_by_threshold.items():
    for dimension in ["E", "S", "G"]:
        sub = dictionary_df[dictionary_df["dimension"].eq(dimension)]
        expanded_count_rows.append({
            "threshold": theta,
            "dimension": dimension,
            "total_terms": len(sub),
            "seed_terms": int(sub["source"].eq("seed").sum()),
            "fasttext_terms": int(sub["source"].eq("fasttext_candidate").sum()),
        })

print("FastText expanded term counts by theta")
display(pd.DataFrame(expanded_count_rows).pivot(index="threshold", columns="dimension", values="fasttext_terms"))

kiwi = Kiwi()
NOUN_TAGS = {"NNG", "NNP", "SL"}


def kiwi_noun_candidates(text):
    terms = []
    current = []

    for token in kiwi.tokenize("" if pd.isna(text) else str(text)):
        form = normalize_term(token.form)
        if token.tag in NOUN_TAGS and len(form) > 1:
            terms.append(form)
            current.append(form)
        else:
            if len(current) >= 2:
                terms.append(normalize_term(" ".join(current)))
            current = []

    if len(current) >= 2:
        terms.append(normalize_term(" ".join(current)))

    return [term for term in terms if term]


appendix_kiwi_doc_df = corpus_df[[
    "stock_code", "company_name", "fiscal_year", "esg_year",
    "rcept_no", "total_word_count", "document_norm",
]].copy()
appendix_kiwi_doc_df["kiwi_terms"] = appendix_kiwi_doc_df["document_norm"].apply(kiwi_noun_candidates)
appendix_kiwi_doc_df["kiwi_document"] = appendix_kiwi_doc_df["kiwi_terms"].apply(
    lambda terms: " ".join(token_key(term) for term in terms)
)
appendix_kiwi_doc_df["kiwi_term_count"] = appendix_kiwi_doc_df["kiwi_terms"].apply(len)

vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    min_df=1,
    norm=None,
)
appendix_tfidf_matrix = vectorizer.fit_transform(appendix_kiwi_doc_df["kiwi_document"])
appendix_vocab = vectorizer.vocabulary_


def prepare_terms(terms):
    prepared = []
    seen = set()
    for term in terms:
        term = normalize_term(term)
        key = token_key(term)
        if term and key in appendix_vocab and term not in seen:
            prepared.append((term, appendix_vocab[key]))
            seen.add(term)
    return prepared


def add_dictionary_scores(score_df, meta_rows, dictionary, threshold, dimension, terms, prefix):
    prepared_terms = prepare_terms(terms)
    term_set = {term for term, _ in prepared_terms}
    tfidf_cols = [col for _, col in prepared_terms]

    count_feature = f"{dimension}_{prefix}_count"
    tfidf_feature = f"{dimension}_{prefix}_tfidf"

    score_df[count_feature] = [
        sum(1 for term in doc_terms if term in term_set)
        for doc_terms in appendix_kiwi_doc_df["kiwi_terms"]
    ]
    score_df[tfidf_feature] = (
        np.asarray(appendix_tfidf_matrix[:, tfidf_cols].sum(axis=1)).ravel()
        if tfidf_cols else np.zeros(len(score_df), dtype=float)
    )

    meta_rows.append({
        "dictionary": dictionary,
        "threshold": threshold,
        "dimension": dimension,
        "prefix": prefix,
        "term_count": len(prepared_terms),
        "dictionary_term_count": len({normalize_term(term) for term in terms if normalize_term(term)}),
    })


appendix_score_df = appendix_kiwi_doc_df[[
    "stock_code", "company_name", "fiscal_year", "esg_year",
    "rcept_no", "total_word_count", "kiwi_term_count",
]].copy()
appendix_meta_rows = []

for dimension in ["E", "S", "G"]:
    seed_terms = seed_dictionary_df.loc[seed_dictionary_df["dimension"].eq(dimension), "candidate_term"].tolist()
    add_dictionary_scores(appendix_score_df, appendix_meta_rows, "seed", np.nan, dimension, seed_terms, "seed")

for theta, dictionary_df in expanded_dictionary_by_threshold.items():
    dictionary = f"fasttext_theta_{threshold_label(theta)}"
    prefix = f"fasttext_{threshold_label(theta)}"
    for dimension in ["E", "S", "G"]:
        terms = dictionary_df.loc[dictionary_df["dimension"].eq(dimension), "candidate_term"].tolist()
        add_dictionary_scores(appendix_score_df, appendix_meta_rows, dictionary, theta, dimension, terms, prefix)

appendix_meta_df = pd.DataFrame(appendix_meta_rows)
for prefix in appendix_meta_df["prefix"].drop_duplicates():
    for score_type in ["count", "tfidf"]:
        appendix_score_df[f"ESG_{prefix}_{score_type}"] = sum(
            appendix_score_df[f"{dimension}_{prefix}_{score_type}"]
            for dimension in ["E", "S", "G"]
        )

company_master = pd.read_csv(COMPANY_MASTER_PATH, dtype={"stock_code": "string"}, encoding="utf-8-sig")
company_master["stock_code"] = company_master["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)
appendix_score_df["stock_code"] = appendix_score_df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)

for col in ["fiscal_year", "esg_year"]:
    company_master[col] = pd.to_numeric(company_master[col], errors="coerce").astype("Int64")
    appendix_score_df[col] = pd.to_numeric(appendix_score_df[col], errors="coerce").astype("Int64")

for col in ["esg_grade", "e_grade", "s_grade", "g_grade"]:
    company_master[f"{col}_num"] = company_master[col].map(GRADE_MAP)

grade_cols = [
    "stock_code", "fiscal_year", "esg_year",
    "esg_grade_num", "e_grade_num", "s_grade_num", "g_grade_num",
]
appendix_analysis_df = appendix_score_df.merge(
    company_master[grade_cols].drop_duplicates(["stock_code", "fiscal_year", "esg_year"]),
    on=["stock_code", "fiscal_year", "esg_year"],
    how="left",
)

appendix_corr_rows = []
for _, meta in appendix_meta_df.iterrows():
    dimension = meta["dimension"]
    grade_col = GRADE_BY_DIMENSION[dimension]
    for score_type in ["count", "tfidf"]:
        feature = f"{dimension}_{meta['prefix']}_{score_type}"
        rho, pvalue, n = safe_spearman(appendix_analysis_df[feature], appendix_analysis_df[grade_col])
        appendix_corr_rows.append({
            "dictionary": meta["dictionary"],
            "threshold": meta["threshold"],
            "dimension": dimension,
            "score_type": score_type,
            "feature": feature,
            "grade_col": grade_col,
            "term_count": int(meta["term_count"]),
            "dictionary_term_count": int(meta["dictionary_term_count"]),
            "spearman_rho": rho,
            "p_value": pvalue,
            "n": n,
        })

for _, group in appendix_meta_df.groupby(["dictionary", "threshold", "prefix"], dropna=False):
    dictionary = group["dictionary"].iloc[0]
    threshold = group["threshold"].iloc[0]
    prefix = group["prefix"].iloc[0]
    term_count = int(group["term_count"].sum())
    dictionary_term_count = int(group["dictionary_term_count"].sum())
    for score_type in ["count", "tfidf"]:
        feature = f"ESG_{prefix}_{score_type}"
        rho, pvalue, n = safe_spearman(appendix_analysis_df[feature], appendix_analysis_df["esg_grade_num"])
        appendix_corr_rows.append({
            "dictionary": dictionary,
            "threshold": threshold,
            "dimension": "ESG",
            "score_type": score_type,
            "feature": feature,
            "grade_col": "esg_grade_num",
            "term_count": term_count,
            "dictionary_term_count": dictionary_term_count,
            "spearman_rho": rho,
            "p_value": pvalue,
            "n": n,
        })

appendix_fasttext_correlation_df = (
    pd.DataFrame(appendix_corr_rows)
    .sort_values(["dimension", "score_type", "threshold"], na_position="first")
    .reset_index(drop=True)
)

print("Appendix in-cell fastText expansion Spearman by theta: rho and p-value")
display(appendix_fasttext_correlation_df)

for score_type in ["count", "tfidf"]:
    expanded_result = appendix_fasttext_correlation_df[
        appendix_fasttext_correlation_df["dictionary"].ne("seed")
        & appendix_fasttext_correlation_df["score_type"].eq(score_type)
    ]

    print(f"Expanded fastText rho by theta - {score_type}")
    display(expanded_result.pivot_table(
        index="threshold",
        columns="dimension",
        values="spearman_rho",
        aggfunc="first",
    ))

    print(f"Expanded fastText p-value by theta - {score_type}")
    display(expanded_result.pivot_table(
        index="threshold",
        columns="dimension",
        values="p_value",
        aggfunc="first",
    ))

print("Seed baseline")
display(appendix_fasttext_correlation_df[appendix_fasttext_correlation_df["dictionary"].eq("seed")])

print("Observed expanded term counts in Kiwi vocabulary")
display(
    appendix_meta_df[appendix_meta_df["dictionary"].ne("seed")]
    .pivot_table(index="threshold", columns="dimension", values="term_count", aggfunc="first")
)

# Incremental explanatory power: does fastText added-only score improve over seed?
import statsmodels.api as sm


def appendix_zscore(series):
    series = series.astype(float)
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return series * 0
    return (series - series.mean()) / std


def appendix_fit_ols(data, y_col, x_cols):
    reg_df = data[[y_col] + x_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(reg_df) < len(x_cols) + 3:
        return None, None, reg_df
    y = reg_df[y_col].astype(float)
    X = reg_df[x_cols].apply(appendix_zscore)
    X = sm.add_constant(X, has_constant="add")
    plain_model = sm.OLS(y, X).fit()
    hc3_model = sm.OLS(y, X).fit(cov_type="HC3")
    return plain_model, hc3_model, reg_df


fasttext_incremental_rows = []
seed_prefix = "seed"
for _, group in appendix_meta_df[appendix_meta_df["dictionary"].ne("seed")].groupby(["dictionary", "threshold", "prefix"], dropna=False):
    dictionary = group["dictionary"].iloc[0]
    threshold = group["threshold"].iloc[0]
    prefix = group["prefix"].iloc[0]
    added_term_count = int(group["term_count"].sum() - appendix_meta_df.loc[appendix_meta_df["prefix"].eq(seed_prefix), "term_count"].sum())
    added_dictionary_term_count = int(group["dictionary_term_count"].sum() - appendix_meta_df.loc[appendix_meta_df["prefix"].eq(seed_prefix), "dictionary_term_count"].sum())

    for score_type in ["count", "tfidf"]:
        seed_feature = f"ESG_{seed_prefix}_{score_type}"
        expanded_feature = f"ESG_{prefix}_{score_type}"
        added_feature = f"ESG_{prefix}_added_only_{score_type}"
        if seed_feature not in appendix_analysis_df.columns or expanded_feature not in appendix_analysis_df.columns:
            continue

        # fastText expanded dictionaries include seed rows, so subtract seed to isolate added terms.
        appendix_analysis_df[added_feature] = appendix_analysis_df[expanded_feature] - appendix_analysis_df[seed_feature]

        base_plain, base_hc3, base_df = appendix_fit_ols(
            appendix_analysis_df,
            "esg_grade_num",
            [seed_feature, "total_word_count"],
        )
        full_plain, full_hc3, full_df = appendix_fit_ols(
            appendix_analysis_df,
            "esg_grade_num",
            [seed_feature, added_feature, "total_word_count"],
        )
        if base_plain is None or full_plain is None:
            continue

        f_stat, f_pvalue, df_diff = full_plain.compare_f_test(base_plain)
        fasttext_incremental_rows.append({
            "dictionary": dictionary,
            "threshold": threshold,
            "score_type": score_type,
            "seed_feature": seed_feature,
            "added_feature": added_feature,
            "n": int(full_plain.nobs),
            "added_term_count": added_term_count,
            "added_dictionary_term_count": added_dictionary_term_count,
            "base_r2": float(base_plain.rsquared),
            "full_r2": float(full_plain.rsquared),
            "delta_r2": float(full_plain.rsquared - base_plain.rsquared),
            "base_adj_r2": float(base_plain.rsquared_adj),
            "full_adj_r2": float(full_plain.rsquared_adj),
            "delta_adj_r2": float(full_plain.rsquared_adj - base_plain.rsquared_adj),
            "base_aic": float(base_plain.aic),
            "full_aic": float(full_plain.aic),
            "delta_aic_full_minus_base": float(full_plain.aic - base_plain.aic),
            "base_bic": float(base_plain.bic),
            "full_bic": float(full_plain.bic),
            "delta_bic_full_minus_base": float(full_plain.bic - base_plain.bic),
            "partial_f": float(f_stat),
            "partial_f_p_value": float(f_pvalue),
            "df_diff": float(df_diff),
            "added_coef_hc3": float(full_hc3.params[added_feature]),
            "added_p_value_hc3": float(full_hc3.pvalues[added_feature]),
        })

appendix_fasttext_incremental_ols_df = (
    pd.DataFrame(fasttext_incremental_rows)
    .sort_values(["score_type", "delta_adj_r2"], ascending=[True, False])
    .reset_index(drop=True)
)

print("FastText added-only incremental OLS over seed baseline")
display(appendix_fasttext_incremental_ols_df)

if not appendix_fasttext_incremental_ols_df.empty:
    print("Best FastText threshold by adjusted R2 gain")
    display(
        appendix_fasttext_incremental_ols_df
        .sort_values("delta_adj_r2", ascending=False)
        .head(10)
    )



Seed query terms


,dimension,query_terms
0,E,18
1,G,18
2,S,18


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.bin:   0%|          | 0.00/7.24G [00:00<?, ?B/s]

FastText expanded term counts by theta


...

FastText added-only incremental OLS over seed baseline


/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:2284: RuntimeWarning: divide by zero encountered in scalar divide
  f_value = (ssr_restr - ssr_full) / df_diff / ssr_full * df_full
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:2284: RuntimeWarning: invalid value encountered in scalar divide
  f_value = (ssr_restr - ssr_full) / df_diff / ssr_full * df_full
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:2284: RuntimeWarning: divide by zero encountered in scalar divide
  f_value = (ssr_restr - ssr_full) / df_diff / ssr_full * df_full
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:2284: RuntimeWarning: invalid value encountered in scalar divide
  f_value = (ssr_restr - ssr_full) / df_diff / ssr_full * df_full
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:2284: RuntimeWarning: divide by zero encountered in scalar divide
  f_value = (

,dictionary,threshold,score_type,seed_feature,added_feature,n,added_term_count,added_dictionary_term_count,base_r2,full_r2,...,full_aic,delta_aic_full_minus_base,base_bic,full_bic,delta_bic_full_minus_base,partial_f,partial_f_p_value,df_diff,added_coef_hc3,added_p_value_hc3
0,fasttext_theta_0_70,0.70,count,ESG_seed_count,ESG_fasttext_0_70_added_only_count,381,3,48,0.300296,0.328125,...,1305.955825,-13.462995,1331.247218,1321.727023,-9.520195,15.615388,0.000093,1.0,0.278734,1.494568e-08
1,fasttext_theta_0_10,0.10,count,ESG_seed_count,ESG_fasttext_0_10_added_only_count,381,1664,21217,0.300296,0.306252,...,1318.161744,-1.257076,1331.247218,1333.932942,2.685724,3.236696,0.072805,1.0,-0.823077,9.812779e-02
2,fasttext_theta_0_20,0.20,count,ESG_seed_count,ESG_fasttext_0_20_added_only_count,381,1664,21217,0.300296,0.306252,...,1318.161744,-1.257076,1331.247218,1333.932942,2.685724,3.236696,0.072805,1.0,-0.823077,9.812779e-02
3,fasttext_theta_0_30,0.30,count,ESG_seed_count,ESG_fasttext_0_30_added_only_count,381,1664,21217,0.300296,0.306252,...,1318.161744,-1.257076,1331.247218,1333.932942,2.685724,3.236696,0.072805,1.0,-0.823077,9.812779e-02
4,fasttext_theta_0_80,0.80,count,ESG_seed_count,ESG_fasttext_0_80_added_only_count,381,0,0,0.300296,0.300296,...,1319.418820,0.000000,1331.247218,1331.247218,0.000000,-inf,NaN,0.0,0.000000,NaN
5,fasttext_theta_0_90,0.90,count,ESG_seed_count,ESG_fasttext_0_90_added_only_count,381,0,0,0.300296,0.300296,...,1319.418820,0.000000,1331.247218,1331.247218,0.000000,-inf,NaN,0.0,0.000000,NaN
6,fasttext_theta_1_00,1.00,count,ESG_seed_count,ESG_fasttext_1_00_added_only_count,381,0,0,0.300296,0.300296,...,1319.418820,0.000000,1331.247218,1331.247218,0.000000,-inf,NaN,0.0,0.000000,NaN
7,fasttext_theta_0_60,0.60,count,ESG_seed_count,ESG_fasttext_0_60_added_only_count,381,17,1106,0.300296,0.300750,...,1321.171178,1.752358,1331.247218,1336.942375,5.695157,0.245122,0.620820,1.0,-0.050349,5.948798e-01
8,fasttext_theta_0_45,0.45,count,ESG_seed_count,ESG_fasttext_0_45_added_only_count,381,500,8445,0.300296,0.300655,...,1321.222997,1.804177,1331.247218,1336.994194,5.746976,0.193817,0.660011,1.0,0.147086,6.733865e-01
9,fasttext_theta_0_55,0.55,count,ESG_seed_count,ESG_fasttext_0_55_added_only_count,381,41,2418,0.300296,0.300417,...,1321.352927,1.934107,1331.247218,1337.124124,5.876906,0.065207,0.798587,1.0,0.025385,7.727485e-01


Best FastText threshold by adjusted R2 gain


,dictionary,threshold,score_type,seed_feature,added_feature,n,added_term_count,added_dictionary_term_count,base_r2,full_r2,...,full_aic,delta_aic_full_minus_base,base_bic,full_bic,delta_bic_full_minus_base,partial_f,partial_f_p_value,df_diff,added_coef_hc3,added_p_value_hc3
0,fasttext_theta_0_70,0.70,count,ESG_seed_count,ESG_fasttext_0_70_added_only_count,381,3,48,0.300296,0.328125,...,1305.955825,-13.462995,1331.247218,1321.727023,-9.520195,15.615388,0.000093,1.0,0.278734,1.494568e-08
13,fasttext_theta_0_70,0.70,tfidf,ESG_seed_tfidf,ESG_fasttext_0_70_added_only_tfidf,381,3,48,0.278778,0.306186,...,1318.197814,-12.761172,1342.787384,1333.969011,-8.818372,14.892834,0.000134,1.0,0.277210,1.649578e-07
14,fasttext_theta_0_10,0.10,tfidf,ESG_seed_tfidf,ESG_fasttext_0_10_added_only_tfidf,381,1664,21217,0.278778,0.286545,...,1328.833808,-2.125178,1342.787384,1344.605005,1.817621,4.104047,0.043485,1.0,-0.893433,3.668139e-02
15,fasttext_theta_0_20,0.20,tfidf,ESG_seed_tfidf,ESG_fasttext_0_20_added_only_tfidf,381,1664,21217,0.278778,0.286545,...,1328.833808,-2.125178,1342.787384,1344.605005,1.817621,4.104047,0.043485,1.0,-0.893433,3.668139e-02
16,fasttext_theta_0_30,0.30,tfidf,ESG_seed_tfidf,ESG_fasttext_0_30_added_only_tfidf,381,1664,21217,0.278778,0.286545,...,1328.833808,-2.125178,1342.787384,1344.605005,1.817621,4.104047,0.043485,1.0,-0.893433,3.668139e-02
1,fasttext_theta_0_10,0.10,count,ESG_seed_count,ESG_fasttext_0_10_added_only_count,381,1664,21217,0.300296,0.306252,...,1318.161744,-1.257076,1331.247218,1333.932942,2.685724,3.236696,0.072805,1.0,-0.823077,9.812779e-02
2,fasttext_theta_0_20,0.20,count,ESG_seed_count,ESG_fasttext_0_20_added_only_count,381,1664,21217,0.300296,0.306252,...,1318.161744,-1.257076,1331.247218,1333.932942,2.685724,3.236696,0.072805,1.0,-0.823077,9.812779e-02
3,fasttext_theta_0_30,0.30,count,ESG_seed_count,ESG_fasttext_0_30_added_only_count,381,1664,21217,0.300296,0.306252,...,1318.161744,-1.257076,1331.247218,1333.932942,2.685724,3.236696,0.072805,1.0,-0.823077,9.812779e-02
17,fasttext_theta_0_65,0.65,tfidf,ESG_seed_tfidf,ESG_fasttext_0_65_added_only_tfidf,381,8,356,0.278778,0.281531,...,1331.501825,0.542840,1342.787384,1347.273023,4.485639,1.444623,0.230147,1.0,0.120216,1.186817e-01
18,fasttext_theta_0_55,0.55,tfidf,ESG_seed_tfidf,ESG_fasttext_0_55_added_only_tfidf,381,41,2418,0.278778,0.280802,...,1331.888302,0.929317,1342.787384,1347.659500,4.872116,1.060933,0.303663,1.0,0.101206,2.454720e-01


### 확장 사전 csv 저장

In [ ]:
# # Save only the selected expanded dictionary candidates to Google Drive.
# # This cell recalculates only fastText theta=0.70 and Kiwi embedding threshold=0.80,
# # so it does not require running the appendix threshold-sweep cells.
# import re
# import subprocess
# import sys
# from collections import Counter
# from pathlib import Path

# import numpy as np
# import pandas as pd
# from IPython.display import display

# try:
#     import fasttext
#     from huggingface_hub import hf_hub_download
# except ImportError:
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fasttext-wheel", "huggingface_hub"])
#     import fasttext
#     from huggingface_hub import hf_hub_download

# try:
#     from kiwipiepy import Kiwi
# except ImportError:
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kiwipiepy"])
#     from kiwipiepy import Kiwi

# try:
#     from sentence_transformers import SentenceTransformer
# except ImportError:
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
#     from sentence_transformers import SentenceTransformer

# try:
#     from tqdm.auto import tqdm
# except ImportError:
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tqdm"])
#     from tqdm.auto import tqdm

# SEED_DICTIONARY_PATH = Path("/content/drive/MyDrive/seed_dictionary.csv")
# OUTPUT_DIR = Path("/content/drive/MyDrive")

# BEST_FASTTEXT_THETA = 0.70
# BEST_KIWI_THRESHOLD = 0.80

# HF_FASTTEXT_REPO_ID = "facebook/fasttext-ko-vectors"
# HF_FASTTEXT_FILENAME = "model.bin"
# FASTTEXT_TOP_K = 500

# KIWI_MODEL_NAME = "dragonkue/BGE-m3-ko"
# KIWI_MIN_TERM_FREQ = 3
# KIWI_MIN_DOC_FREQ = 2
# KIWI_MAX_CANDIDATES = 30000
# KIWI_BATCH_SIZE = 64


# def normalize_term(value):
#     text = "" if pd.isna(value) else str(value)
#     text = re.sub(r"\s+", " ", text).strip()
#     return text.strip(" \t\r\n\"'`.,;:()[]{}<>")


# def threshold_label(theta):
#     return f"{float(theta):.2f}".replace(".", "_")


# def split_seed_terms(row):
#     values = [row.get("seed_term", "")]
#     pattern = row.get("pattern", "")
#     if pd.notna(pattern):
#         values.extend(str(pattern).split("|"))

#     terms = []
#     seen = set()
#     for value in values:
#         term = normalize_term(value)
#         if term and term.lower() != "nan" and term not in seen:
#             terms.append(term)
#             seen.add(term)
#     return terms


# def build_seed_query_df(seed_df):
#     rows = []
#     for _, row in seed_df.iterrows():
#         dimension = normalize_term(row.get("dimension", ""))
#         if dimension not in {"E", "S", "G"}:
#             continue

#         seed_term = normalize_term(row.get("seed_term", ""))
#         for query_term in split_seed_terms(row):
#             rows.append({
#                 "dimension": dimension,
#                 "seed_term": seed_term,
#                 "query_term": query_term,
#                 "query_text": query_term.replace("_", " "),
#             })

#     return (
#         pd.DataFrame(rows)
#         .drop_duplicates(["dimension", "query_term"])
#         .reset_index(drop=True)
#     )


# def build_fasttext_dictionary(seed_query_df, theta):
#     print(f"Downloading/loading fastText model from {HF_FASTTEXT_REPO_ID}/{HF_FASTTEXT_FILENAME}...")
#     model_path = hf_hub_download(repo_id=HF_FASTTEXT_REPO_ID, filename=HF_FASTTEXT_FILENAME)
#     print(f"Loading fastText model: {model_path}")
#     fasttext_model = fasttext.load_model(model_path)
#     print("fastText model loaded. Collecting nearest neighbors...")

#     candidate_rows = []
#     seed_iter = tqdm(
#         seed_query_df.iterrows(),
#         total=len(seed_query_df),
#         desc=f"fastText theta>={theta:.2f}",
#         unit="seed",
#     )
#     for _, row in seed_iter:
#         query_term = row["query_term"]
#         try:
#             neighbors = fasttext_model.get_nearest_neighbors(query_term, k=FASTTEXT_TOP_K)
#         except Exception:
#             continue

#         for similarity, candidate in neighbors:
#             candidate_term = normalize_term(candidate.replace("_", " "))
#             if not candidate_term or similarity < theta:
#                 continue
#             candidate_rows.append({
#                 "dictionary": f"fasttext_theta_{threshold_label(theta)}",
#                 "threshold": theta,
#                 "dimension": row["dimension"],
#                 "seed_term": row["seed_term"],
#                 "query_term": query_term,
#                 "candidate_term": candidate_term,
#                 "source": "fasttext_neighbor",
#                 "similarity": float(similarity),
#             })

#     seed_rows = (
#         seed_query_df.rename(columns={"query_term": "candidate_term"})[
#             ["dimension", "seed_term", "candidate_term"]
#         ]
#         .drop_duplicates(["dimension", "candidate_term"])
#         .copy()
#     )
#     seed_rows["dictionary"] = f"fasttext_theta_{threshold_label(theta)}"
#     seed_rows["threshold"] = theta
#     seed_rows["query_term"] = seed_rows["candidate_term"]
#     seed_rows["source"] = "seed"
#     seed_rows["similarity"] = 1.0

#     print(f"fastText candidates kept at theta>={theta:.2f}: {len(candidate_rows):,}")
#     candidate_df = pd.DataFrame(candidate_rows)
#     dictionary_df = pd.concat([seed_rows, candidate_df], ignore_index=True, sort=False)
#     return (
#         dictionary_df
#         .drop_duplicates(["dimension", "candidate_term"], keep="first")
#         .sort_values(["dimension", "source", "similarity", "candidate_term"], ascending=[True, False, False, True])
#         .reset_index(drop=True)
#     )


# kiwi = Kiwi()
# NOUN_TAGS = {"NNG", "NNP", "SL"}


# def kiwi_noun_candidates(text):
#     terms = []
#     current = []
#     for token in kiwi.tokenize("" if pd.isna(text) else str(text)):
#         form = normalize_term(token.form)
#         if token.tag in NOUN_TAGS and len(form) > 1:
#             terms.append(form)
#             current.append(form)
#         else:
#             if len(current) >= 2:
#                 terms.append(normalize_term(" ".join(current)))
#             current = []

#     if len(current) >= 2:
#         terms.append(normalize_term(" ".join(current)))

#     return [term for term in terms if term]


# def encode_texts(model, texts, batch_size):
#     return model.encode(
#         list(texts),
#         batch_size=batch_size,
#         convert_to_numpy=True,
#         normalize_embeddings=True,
#         show_progress_bar=True,
#     )


# def build_kiwi_dictionary(seed_query_df, threshold):
#     if "corpus_df" not in globals():
#         raise RuntimeError("corpus_df is required. Run the corpus-building cells before this save cell.")
#     if "document_norm" not in corpus_df.columns:
#         raise RuntimeError("corpus_df must contain a document_norm column.")

#     doc_terms = corpus_df["document_norm"].apply(kiwi_noun_candidates)
#     term_frequency = Counter(term for terms in doc_terms for term in terms)
#     doc_frequency = Counter(term for terms in doc_terms for term in set(terms))

#     seed_terms = set(seed_query_df["query_term"])
#     candidate_terms = [
#         term for term, freq in term_frequency.most_common()
#         if freq >= KIWI_MIN_TERM_FREQ
#         and doc_frequency[term] >= KIWI_MIN_DOC_FREQ
#         and term not in seed_terms
#     ][:KIWI_MAX_CANDIDATES]

#     import torch
#     device = "cuda" if torch.cuda.is_available() else "cpu"
#     batch_size = KIWI_BATCH_SIZE if device == "cuda" else min(16, KIWI_BATCH_SIZE)
#     model = SentenceTransformer(KIWI_MODEL_NAME, device=device)

#     seed_embeddings = encode_texts(model, seed_query_df["query_text"], batch_size)
#     dimension_indices = {
#         dimension: list(indices)
#         for dimension, indices in seed_query_df.groupby("dimension").groups.items()
#     }

#     candidate_rows = []
#     for start in range(0, len(candidate_terms), batch_size):
#         batch_terms = candidate_terms[start:start + batch_size]
#         batch_embeddings = encode_texts(model, batch_terms, batch_size)
#         similarity_matrix = batch_embeddings @ seed_embeddings.T

#         for row_i, candidate_term in enumerate(batch_terms):
#             similarities = similarity_matrix[row_i]
#             best_i = int(np.argmax(similarities))
#             best_seed = seed_query_df.iloc[best_i]
#             best_dimension = best_seed["dimension"]
#             best_dimension_score = float(similarities[best_i])

#             if best_dimension_score < threshold:
#                 continue

#             dimension_scores = {
#                 dimension: float(np.max(similarities[indices]))
#                 for dimension, indices in dimension_indices.items()
#             }
#             ordered_scores = sorted(dimension_scores.values(), reverse=True)
#             second_dimension_score = ordered_scores[1] if len(ordered_scores) > 1 else np.nan

#             candidate_rows.append({
#                 "dictionary": "kiwi_embedding_expanded",
#                 "threshold": threshold,
#                 "dimension": best_dimension,
#                 "candidate_term": candidate_term,
#                 "source": "kiwi_embedding_candidate",
#                 "seed_term": best_seed["seed_term"],
#                 "best_query_term": best_seed["query_term"],
#                 "best_dimension_score": best_dimension_score,
#                 "second_dimension_score": second_dimension_score,
#                 "dimension_margin": best_dimension_score - second_dimension_score if not np.isnan(second_dimension_score) else np.nan,
#                 "term_frequency": int(term_frequency[candidate_term]),
#                 "doc_frequency": int(doc_frequency[candidate_term]),
#             })

#     kiwi_seed_rows = (
#         seed_query_df.rename(columns={"query_term": "candidate_term"})[
#             ["dimension", "seed_term", "candidate_term"]
#         ]
#         .drop_duplicates(["dimension", "candidate_term"])
#         .copy()
#     )
#     kiwi_seed_rows["dictionary"] = "kiwi_embedding_expanded"
#     kiwi_seed_rows["threshold"] = threshold
#     kiwi_seed_rows["source"] = "seed"
#     kiwi_seed_rows["best_dimension_score"] = 1.0
#     kiwi_seed_rows["second_dimension_score"] = pd.NA
#     kiwi_seed_rows["dimension_margin"] = pd.NA
#     kiwi_seed_rows["best_query_term"] = kiwi_seed_rows["candidate_term"]
#     kiwi_seed_rows["term_frequency"] = pd.NA
#     kiwi_seed_rows["doc_frequency"] = pd.NA

#     kiwi_columns = [
#         "dictionary", "threshold", "dimension", "candidate_term", "source",
#         "seed_term", "best_query_term", "best_dimension_score",
#         "second_dimension_score", "dimension_margin", "term_frequency", "doc_frequency",
#     ]

#     return (
#         pd.concat([kiwi_seed_rows, pd.DataFrame(candidate_rows)], ignore_index=True, sort=False)
#         .drop_duplicates(["dimension", "candidate_term"], keep="first")
#         .reindex(columns=kiwi_columns)
#         .sort_values(["dimension", "source", "best_dimension_score", "candidate_term"], ascending=[True, False, False, True])
#         .reset_index(drop=True)
#     )


# seed_source_df = pd.read_csv(SEED_DICTIONARY_PATH, encoding="utf-8-sig")
# seed_query_df = build_seed_query_df(seed_source_df)

# # Candidate 1: fastText expansion, selected by the appendix incremental OLS result.
# best_fasttext_dictionary_df = build_fasttext_dictionary(seed_query_df, BEST_FASTTEXT_THETA)
# best_fasttext_dictionary_df["selected_by"] = "best_incremental_ols_candidate"
# best_fasttext_dictionary_df["selection_note"] = "fastText theta=0.70 had the strongest incremental explanatory power over the seed baseline in the appendix"

# fasttext_output_path = OUTPUT_DIR / "candidate_fasttext_theta_0_70_dictionary.csv"
# best_fasttext_dictionary_df.to_csv(fasttext_output_path, index=False, encoding="utf-8-sig")

# # Candidate 2: Kiwi embedding expansion, selected by the appendix incremental OLS result.
# best_kiwi_dictionary_df = build_kiwi_dictionary(seed_query_df, BEST_KIWI_THRESHOLD)
# best_kiwi_dictionary_df["selected_by"] = "best_incremental_ols_candidate"
# best_kiwi_dictionary_df["selection_note"] = "Kiwi embedding threshold=0.80 had the strongest incremental explanatory power over the seed baseline in the appendix"

# kiwi_output_path = OUTPUT_DIR / "candidate_kiwi_embedding_threshold_0_80_dictionary.csv"
# best_kiwi_dictionary_df.to_csv(kiwi_output_path, index=False, encoding="utf-8-sig")

# print("Saved candidate expanded dictionaries:")
# print(f"- fastText theta={BEST_FASTTEXT_THETA:.2f}: {fasttext_output_path}")
# print(f"- Kiwi embedding threshold={BEST_KIWI_THRESHOLD:.2f}: {kiwi_output_path}")

# print("\nfastText terms by dimension/source")
# display(best_fasttext_dictionary_df.groupby(["dimension", "source"]).size().rename("terms").reset_index())

# print("Kiwi embedding terms by dimension/source")
# display(best_kiwi_dictionary_df.groupby(["dimension", "source"]).size().rename("terms").reset_index())
